# Assignment 2B - RAG Pipeline

**Domain:** Type 2 diabetes management. **Total:** 20 marks.

| Part | Scope | Marks |
| --- | --- | --- |
| A | Dense search and chunk size | 5 |
| B | Dense, BM25, and hybrid search | 4 |
| C | Cross-encoder reranking | 4 |
| D | Grounded generation and LLM evaluation | 7 |

| Role | Model / method |
| --- | --- |
| Generator | Assignment 1 Mistral-7B-v0.1 + CPT + QLoRA adapter, then Qwen/Qwen2.5-7B-Instruct (D1 switch) |
| Embeddings | BAAI/bge-base-en-v1.5 (512-token input limit) |
| Dense index | FAISS IndexFlatIP with normalized embeddings |
| Keyword / hybrid | BM25 / reciprocal rank fusion |
| Reranker | cross-encoder/ms-marco-MiniLM-L-6-v2 |
| Judge | meta-llama/Llama-3.1-8B-Instruct |

## Setup

Run on the CUDA lab server with the Assignment 1 cleaned corpus, instruction dataset, CPT checkpoint, and adapter from the same run.

1. Configure the input paths, particularly `DATASET_PATH`.
   Set `HF_TOKEN` to a Hugging Face token with access to `meta-llama/Llama-3.1-8B-Instruct` (the D2 judge).
2. Review `SPAN_REVIEW` on the first run; approved queries and spans are saved for reuse.
3. After generation, complete `SPOT_REVIEW` and `FAILURE_REVIEW` using the displayed evidence.
4. Keep all outputs.

The notebook pauses at missing inputs or unapproved annotations rather than scoring unverified data.

In [1]:
# Detect the runtime, configure cache storage, and inspect the available GPU.
import importlib.util
import os
from types import SimpleNamespace


def detect_env():
    """Return "kubeflow", "colab" or "local". Kubeflow is checked first."""
    if ("NB_PREFIX" in os.environ or "KUBERNETES_SERVICE_HOST" in os.environ
            or os.path.isdir("/home/jovyan/data")):
        return "kubeflow"
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "colab"
    except ModuleNotFoundError:
        pass
    return "local"


def setup():
    """Pick the storage folder, set the Hugging Face cache, and read the device and dtype."""
    env = detect_env()
    candidates = {"colab": ["/content"], "kubeflow": ["/home/jovyan/assessment", "/home/jovyan/data"],
                  "local": [os.path.expanduser("~/lab-data"), os.getcwd()]}[env]
    root = os.getcwd()
    # Test write access before choosing where large downloaded model files will live.
    for path in candidates:
        try:
            os.makedirs(path, exist_ok=True)
            probe = os.path.join(path, ".write_test")
            open(probe, "w").close()
            os.remove(probe)
            root = path
            break
        except Exception:
            continue

    # Set the cache location before importing Hugging Face model libraries.
    cache_root = os.path.join(root, "hf-cache")
    os.environ["HF_HOME"] = cache_root
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

    device, dtype, gpu_name, gpu_mem = "cpu", "float32", None, 0.0
    try:
        import torch
        if torch.cuda.is_available():
            device = "cuda"
            gpu_name = torch.cuda.get_device_name(0)
            gpu_mem = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
            # Use BF16 only when the GPU supports it; otherwise use FP16.
            dtype = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
    except Exception:
        pass

    state = SimpleNamespace(DEVICE=device, DTYPE=dtype, IS_GPU=(device == "cuda"))
    print(f"Environment: {env}; GPU: {gpu_name or 'none'} ({gpu_mem} GB)")
    print(f"HF cache: {cache_root}")
    return state


L = setup()

Environment: kubeflow; GPU: NVIDIA A100-SXM4-80GB (85.0 GB)
HF cache: /home/jovyan/assessment/hf-cache


In [2]:
# Set the Assignment 1 input paths and check that the required artifacts exist.
import json
import random
import re
from pathlib import Path

# SEED is shared with NumPy and Torch later to make random initialization repeatable.
SEED = 42
random.seed(SEED)

# Reuse the corpus and trained generator artifacts from the same Assignment 1 run.
ASS1 = Path("../Ass-1").resolve()
CORPUS_DIR = ASS1 / "domain_corpus"  # Cleaned documents to chunk and search.
CPT_DIR = ASS1 / "persistent" / "cpt_ckpt"  # Domain-adapted base weights for the generator.
ADAPTER_DIR = ASS1 / "persistent" / "adapter"  # Trained QLoRA updates and saved tokenizer.
DATASET_PATH = ASS1 / "instruction_dataset.jsonl"  # Its last 80 rows are the evaluation split.

# DATA_DIR keeps reviewed annotations; OUT_DIR keeps result tables, answers, and audits.
DATA_DIR, OUT_DIR = Path("data"), Path("outputs")
for folder in (DATA_DIR, OUT_DIR):
    folder.mkdir(exist_ok=True)


# Check the inputs before the models are loaded.
# 2B needs the corpus, instruction dataset, checkpoint and adapter.
checks = {
    "cleaned corpus .txt files": CORPUS_DIR.is_dir() and bool(list(CORPUS_DIR.glob("*.txt"))),
    "instruction dataset": DATASET_PATH.is_file(),
    "CPT checkpoint": CPT_DIR.is_dir(),
    "QLoRA adapter": ADAPTER_DIR.is_dir(),
}
for label, present in checks.items():
    print(f"{'OK' if present else 'MISSING':8} {label}")
if not all(checks.values()):
    raise FileNotFoundError("Assignment 1 inputs are missing on this server.")

OK       cleaned corpus .txt files
OK       instruction dataset
OK       CPT checkpoint
OK       QLoRA adapter


### Dependencies and Utilities

Install missing packages and record package versions.


In [3]:
# Install missing dependencies, seed random generators, and record package versions.
import importlib.metadata
import importlib.util
import subprocess
import sys

PACKAGES = {
    "numpy": "numpy",
    "torch": "torch",
    "packaging": "packaging",
    "sentence-transformers": "sentence_transformers",
    "faiss-cpu": "faiss",
    "rank-bm25": "rank_bm25",
    "transformers": "transformers",
    "peft": "peft",
    "bitsandbytes": "bitsandbytes",
    "accelerate": "accelerate",
}
missing = [package for package, module in PACKAGES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

import numpy as np
import torch

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
L.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
L.IS_GPU = L.DEVICE == "cuda"
L.DTYPE = ("bfloat16" if torch.cuda.is_bf16_supported() else "float16") if L.IS_GPU else "float32"
versions = {package: importlib.metadata.version(package) for package in PACKAGES}
(OUT_DIR / "package_versions.json").write_text(json.dumps(versions, indent=2), encoding="utf-8")
for package, version in versions.items():
    print(f"{package}: {version}")
print(f"Device: {L.DEVICE}; compute dtype: {L.DTYPE}")

numpy: 2.4.6
torch: 2.5.1+cu124
packaging: 24.2
sentence-transformers: 3.3.1
faiss-cpu: 1.9.0
rank-bm25: 0.2.2
transformers: 4.46.3
peft: 0.13.2
bitsandbytes: 0.44.1
accelerate: 1.1.1
Device: cuda; compute dtype: bfloat16


## Helpers and Shared Functions

Table printing, model-load timing, and the functions shared by Parts A to D. They are defined here, so each part's cells only call them.

In [4]:
# Helpers: table printing, model-load timing, and the functions used by Parts A to D.
import time
from contextlib import contextmanager


# Table helpers: print a plain-text table, or render it as Markdown.


def print_table(headers, rows, title=None):
    """Print a text table. Cells should be strings."""
    # The first column is left-aligned; the other columns are right-aligned.
    body = list(rows)
    widths = [
        max(len(header), max(len(row[column]) for row in body))
        for column, header in enumerate(headers)
    ]
    gap = "   "
    table_width = sum(widths) + len(gap) * (len(widths) - 1)

    def format_row(values):
        return gap.join(
            value.ljust(width) if column == 0 else value.rjust(width)
            for column, (value, width) in enumerate(zip(values, widths))
        )

    if title:
        print("=" * table_width)
        print(title)
    print("=" * table_width)
    print(format_row(headers))
    print("-" * table_width)
    for row in rows:
        print(format_row(row))
    print("=" * table_width)


def show_md_table(headers, rows, title=None):
    """Render a wrapping Markdown table."""
    from IPython.display import Markdown, display

    def format_cell(text):
        # Flatten whitespace and replace pipes so text stays in its own column.
        return " ".join(str(text).split()).replace("|", "/")

    markdown = f"**{title}**\n\n" if title else ""
    markdown += "| " + " | ".join(headers) + " |\n"
    markdown += "| " + " | ".join("---" for _ in headers) + " |\n"
    markdown += "\n".join(
        "| " + " | ".join(format_cell(value) for value in row) + " |"
        for row in rows
    )
    display(Markdown(markdown))


# Timing helper: report how long a model-loading step takes, and whether it failed.
@contextmanager
def working(label):
    started = time.perf_counter()
    print(f"{label}...")
    try:
        yield
    except Exception:
        print(f"{label}: failed after {time.perf_counter() - started:.1f}s")
        raise
    else:
        print(f"{label}: completed in {time.perf_counter() - started:.1f}s")


# Other helpers, grouped by the part that uses them.
def dense_search(chunks, queries, k):
    """Return ranked chunks with scores, reusing each chunk set's FAISS index."""
    cache_key = (id(chunks), len(chunks))
    if cache_key not in _INDEX_CACHE:
        started = time.perf_counter()
        # Unit-length vectors make inner product equivalent to cosine similarity.
        vectors = embed_model.encode(
            [chunk["text"] for chunk in chunks], batch_size=EMBED_BATCH,
            normalize_embeddings=True, convert_to_numpy=True,
        ).astype("float32")
        index = faiss.IndexFlatIP(vectors.shape[1])
        index.add(vectors)
        _INDEX_CACHE[cache_key] = (chunks, index, time.perf_counter() - started)
    _, index, _ = _INDEX_CACHE[cache_key]

    # Queries must use the same normalization as the indexed chunks.
    query_vectors = embed_model.encode(
        queries, batch_size=EMBED_BATCH,
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype("float32")
    scores, ids = index.search(query_vectors, k)
    results = []
    for query_index in range(len(queries)):
        # FAISS uses -1 for unfilled slots when k exceeds the number of chunks.
        results.append([
            {"rank": rank_index + 1, "chunk_id": chunks[chunk_index]["chunk_id"],
             "score": float(score), "text": chunks[chunk_index]["text"]}
            for rank_index, (score, chunk_index) in enumerate(zip(scores[query_index], ids[query_index]))
            if chunk_index >= 0
        ])
    return results


def chunk_corpus(size):
    """Split corpus files into non-overlapping chunks of up to size embedding tokens."""
    chunks = []
    for path in sorted(CORPUS_DIR.glob("*.txt")):
        text = path.read_text(encoding="utf-8")
        # Character offsets preserve the original wording rather than reconstructing decoded tokens.
        offsets = embed_tok(text, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
        # A full-size stride gives non-overlapping chunks; the final chunk may be shorter.
        for start_offset in range(0, len(offsets), size):
            chunk_offsets = offsets[start_offset:start_offset + size]
            chunks.append({
                "chunk_id": f"{path.stem}#{start_offset // size}",
                "doc": path.name,
                "text": text[chunk_offsets[0][0]:chunk_offsets[-1][1]],
                "n_tokens": len(chunk_offsets),
            })
    return chunks


def index_build_seconds(chunks):
    """Return the time spent embedding and indexing this chunk set."""
    if (id(chunks), len(chunks)) not in _INDEX_CACHE:
        dense_search(chunks, [QUERIES[0]["query"]], 1)
    return _INDEX_CACHE[(id(chunks), len(chunks))][2]


def build_generator_prompt(tokenizer, question, chunk_texts):
    if len(chunk_texts) != 3:
        raise ValueError("Generation requires exactly three chunks.")
    content = GEN_PROMPT.format(chunk_1=chunk_texts[0], chunk_2=chunk_texts[1],
                                chunk_3=chunk_texts[2], question=question)
    return tokenizer.apply_chat_template([{"role": "user", "content": content}],
                                         tokenize=False, add_generation_prompt=True)


def evaluate(search_fn, k=5):
    """Measure mean query latency and hit rates, excluding one warm-up call."""
    # Exclude the warm-up so one-time setup does not inflate measured query latency.
    search_fn(QUERIES[0]["query"], k)
    latencies_ms, ranked = [], {}
    for item in QUERIES:
        started = time.perf_counter()
        ranked[item["qid"]] = search_fn(item["query"], k)
        latencies_ms.append((time.perf_counter() - started) * 1000)

    def hit_at(limit):
        """Percentage of queries with the answer span in the top results."""
        # Multiple matching chunks still contribute only one successful query.
        hits = sum(
            any(is_hit(result["text"], item["span"]) for result in ranked[item["qid"]][:limit])
            for item in QUERIES
        )
        return 100 * hits / len(QUERIES)

    return {
        "latency_ms": sum(latencies_ms) / len(latencies_ms),
        "hit1": hit_at(1),
        "hit3": hit_at(3),
        "ranked": ranked,
    }


def dense_fn(query, k):
    return dense_search(chunks, [query], k)[0]


def bm25_tokens(text):
    """Split text into lowercase words and numbers."""
    # Keep clinical terms such as HbA1c and SGLT2 together as single tokens.
    return re.findall(r"[a-z0-9]+", text.lower())


def bm25_fn(query, k):
    """Return the top k chunks in the same format as dense_search."""
    scores = bm25.get_scores(bm25_tokens(query))
    # Negating scores puts the highest-scoring chunks first.
    top_indices = np.argsort(-scores)[:k]
    return [
        {"rank": rank_index + 1, "chunk_id": chunks[chunk_index]["chunk_id"],
         "score": float(scores[chunk_index]), "text": chunks[chunk_index]["text"]}
        for rank_index, chunk_index in enumerate(top_indices)
    ]


def rrf_fuse(dense_ids, bm25_ids, k=60, top=5):
    """Sum 1 / (k + rank) for each chunk across the two ranked lists."""
    scores = {}
    for ranked_ids in (dense_ids, bm25_ids):
        # Combine ranks because dense and BM25 scores use different scales.
        for rank, chunk_id in enumerate(ranked_ids, start=1):
            scores[chunk_id] = scores.get(chunk_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda item: -item[1])[:top]


def hybrid_fn(query, k):
    """Fuse the top 20 results from each retriever and return the top k chunks."""
    # Take the top 20 from each method, so a chunk ranked low in one list can still reach the top five.
    dense_top = dense_fn(query, 20)
    bm25_top = bm25_fn(query, 20)
    fused = rrf_fuse(
        [result["chunk_id"] for result in dense_top],
        [result["chunk_id"] for result in bm25_top],
        top=k,
    )
    return [
        {"rank": rank_index + 1, "chunk_id": chunk_id,
         "score": score, "text": chunk_by_id[chunk_id]["text"]}
        for rank_index, (chunk_id, score) in enumerate(fused)
    ]


def rank_of(method, qid, span):
    """Return the first matching chunk's top-five rank, or '-' if none matches."""
    for result in b_results[method]["ranked"][qid]:
        if is_hit(result["text"], span):
            return str(result["rank"])
    return "-"


def answer_rank(method, item):
    value = rank_of(method, item["qid"], item["span"])
    return int(value) if value.isdigit() else 99


def rerank_top3(query):
    """Return the top three before and after reranking, plus scoring time in seconds."""
    candidates = best_fn(query, 10)
    # Score each query and chunk together, rather than comparing separate embeddings.
    pairs = [(query, chunk["text"]) for chunk in candidates]
    started = time.perf_counter()
    scores = ce.predict(pairs)
    if L.IS_GPU:
        # GPU scoring must finish before the timer stops.
        torch.cuda.synchronize()
    seconds = time.perf_counter() - started
    order = sorted(range(len(candidates)), key=lambda index: -scores[index])[:3]
    reranked = [
        dict(candidates[index], rank=rank_index + 1, ce_score=float(scores[index]))
        for rank_index, index in enumerate(order)
    ]
    return candidates[:3], reranked, seconds


def load_nf4(model_id):
    """Load a Hugging Face model and tokenizer in NF4 for greedy (temperature 0) decoding."""
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb_config, **{DTYPE_KW: COMPUTE_DTYPE}, device_map={"": 0},
    ).eval()
    # Clear sampling defaults so greedy decoding runs without warnings.
    model.generation_config.temperature = None
    model.generation_config.top_p = None
    model.generation_config.top_k = None
    return model, tokenizer


def clean_answer(raw, stops):
    # Cut the reply where the model starts a new instruction, and drop a leading "Response:" label.
    text = raw
    for stop in stops:
        text = text.split(stop, 1)[0]
    if "Response:" in text:
        text = text.split("Response:", 1)[1]
    # The refusal, with at most one garbled word before it, is returned as exactly that sentence.
    if re.fullmatch(r"\S*\s*(not\s+)?found in the documents\.?", text.strip(), flags=re.I):
        return NOT_FOUND
    return text.strip()


def generate_answer(model, tokenizer, question, chunk_texts, stops=()):
    # Every generator gets the same GEN_PROMPT and the same three numbered chunks.
    prompt = build_generator_prompt(tokenizer, question, chunk_texts)
    encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=False)
    prompt_tokens = encoded["input_ids"].shape[1]
    # Refuse an oversized prompt instead of silently dropping evidence.
    if prompt_tokens + GEN_MAX_NEW_TOKENS > GEN_CONTEXT_LIMIT:
        raise ValueError(f"Prompt ({prompt_tokens}) plus answer budget ({GEN_MAX_NEW_TOKENS}) "
                         f"exceeds the {GEN_CONTEXT_LIMIT}-token context; no chunks were truncated.")
    with torch.no_grad():
        # Greedy decoding (temperature 0) makes the answers repeatable.
        generated = model.generate(**encoded.to(model.device), max_new_tokens=GEN_MAX_NEW_TOKENS,
                                   do_sample=False, pad_token_id=tokenizer.eos_token_id)
    # Decode only the new tokens, not the prompt.
    raw = tokenizer.decode(generated[0][prompt_tokens:], skip_special_tokens=True)
    return clean_answer(raw, stops)


def is_not_found(answer):
    return answer.strip().rstrip(".").strip() == NOT_FOUND


def has_valid_citation(answer):
    # Citation numbers 1-3 show the format only; the judge checks whether the chunks support the claims.
    numbers = [int(number) for number in CITE.findall(answer)]
    return bool(numbers) and all(1 <= number <= 3 for number in numbers)


def cited_numbers(answer):
    return ",".join(sorted(set(CITE.findall(answer)))) or "-"


def judge_generate_raw(prompt_text):
    formatted_prompt = judge_tok.apply_chat_template(
        [{"role": "user", "content": prompt_text}], tokenize=False, add_generation_prompt=True,
    )
    model_inputs = judge_tok(formatted_prompt, return_tensors="pt", add_special_tokens=False, truncation=False)
    # Reserve reply tokens without dropping any reference or evidence text.
    if model_inputs["input_ids"].shape[1] + JUDGE_MAX_NEW_TOKENS > JUDGE_CONTEXT_LIMIT:
        raise ValueError("Judge prompt plus reply budget exceeds context capacity; no evidence was truncated.")
    model_inputs = model_inputs.to(judge_model.device)
    with torch.no_grad():
        # Temperature 0 (greedy) so the judge gives the same verdict every time.
        generated = judge_model.generate(**model_inputs, max_new_tokens=JUDGE_MAX_NEW_TOKENS,
                                         do_sample=False, pad_token_id=judge_tok.eos_token_id)
    return judge_tok.decode(generated[0][model_inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def parse_verdict(text):
    # Allow Markdown decoration, but reject missing or duplicate label lines.
    cleaned = re.sub(r"[<>*]", "", text)
    fields = {}
    for label in ("CORRECT", "SUPPORTED", "REASON"):
        matches = re.findall(rf"^\s*{label}\s*:\s*([^\n]+)", cleaned, flags=re.I | re.M)
        fields[label] = matches[0].strip() if len(matches) == 1 else ""
    correct = fields["CORRECT"].rstrip(".").upper()
    supported = fields["SUPPORTED"].rstrip(".").upper()
    return {
        "correct": correct if correct in {"CORRECT", "PARTLY CORRECT", "WRONG"} else "UNPARSED",
        "supported": supported if supported in {"YES", "NO"} else "UNPARSED",
        "reason": fields["REASON"],
    }


def valid_verdict(verdict):
    # A NO verdict needs a quoted unsupported claim, not just a label.
    return (
        verdict["correct"] in {"CORRECT", "PARTLY CORRECT", "WRONG"}
        and verdict["supported"] in {"YES", "NO"}
        and bool(verdict["reason"].strip())
        and (verdict["supported"] != "NO" or bool(re.search(r'"[^"\n]+"', verdict["reason"])))
    )


def has_claim(answer):
    # Citation markers, and any text in brackets, do not count as an answer sentence.
    return bool(re.sub(r"\[[^\]]*\]", "", answer).strip(" .,;:"))


def judge_generate(prompt_text, answer_has_claim):
    retry_prompt = prompt_text
    for attempt in range(3):
        raw = judge_generate_raw(retry_prompt)
        parsed = parse_verdict(raw)
        # An answer with no claim must be WRONG; any other label is a format failure.
        parsed_ok = valid_verdict(parsed) and (answer_has_claim or parsed["correct"] == "WRONG")
        # Keep failed attempts too, so format errors remain visible in the audit.
        JUDGE_ATTEMPTS.append({
            "attempt": attempt + 1,
            "prompt": retry_prompt,
            "raw": raw,
            "valid": parsed_ok,
        })
        (OUT_DIR / "judge_attempts.json").write_text(
            json.dumps(JUDGE_ATTEMPTS, indent=2, ensure_ascii=False), encoding="utf-8",
        )
        if parsed_ok:
            return raw
        # Retry with the original evidence; do not fill in labels by hand.
        retry_prompt = (
            prompt_text + "\n\nYour previous response did not meet the required format:\n" + raw
            + '\nRe-evaluate the original evidence and return exactly three labeled lines: '
            'CORRECT, SUPPORTED, REASON. Use only the allowed labels, include a nonempty reason, '
            'and quote the unsupported statement in double quotes when SUPPORTED is NO.'
        )
    raise ValueError("Judge response remains invalid after three attempts; inspect outputs/judge_attempts.json.")


def failed(qid):
    judge = verdicts[qid]
    human = SPOT_REVIEW.get(qid, {})
    return (judge["correct"] == "WRONG" or judge["supported"] == "NO"
            or human.get("correct") == "WRONG" or human.get("supported") == "NO")

print("Helpers ready: table and timing helpers, then the functions used by Parts A to D.")


Helpers ready: table and timing helpers, then the functions used by Parts A to D.


## Evaluation Queries and Answer Spans

Use ten distinct queries and unchanged expected answers from Assignment 1's evaluation split: the last 80 rows of `instruction_dataset.jsonl`.

Word overlap proposes candidate spans; questions may share source documents. Human review must confirm that each span answers its question. Approved queries, references, spans, and input hashes are frozen for later runs. Selection favors answer overlap, so conclusions apply only to this small benchmark.

In [5]:
# Before you start: load or propose the ten queries and their answer spans.
import hashlib

# Load the instruction dataset rows (question and reference answer).
with open(DATASET_PATH, encoding="utf-8") as handle:
    dataset_rows = [json.loads(line) for line in handle if line.strip()]
# Assignment 1 wrote the train rows first and the eval rows last, so the eval split is the last 80.
if len(dataset_rows) != 400:
    raise ValueError("The last-80 evaluation split expects 400 dataset rows; verify Assignment 1.")
eval_rows = dataset_rows[-80:]
if len(eval_rows) < 10:
    raise ValueError("At least ten evaluation queries are required.")
for row in eval_rows:
    if not all(isinstance(row.get(field), str) and row[field].strip()
               for field in ("instruction", "response")):
        raise ValueError("Evaluation rows need nonempty instruction and response strings.")

# Sorting makes tied candidate matches repeatable across runs.
corpus_files = sorted(CORPUS_DIR.glob("*.txt"))
# Invalidate saved reviews if the split or corpus has changed.
INPUT_HASHES = {"split": hashlib.sha256(DATASET_PATH.read_bytes()).hexdigest(),
                "corpus": {path.name: hashlib.sha256(path.read_bytes()).hexdigest()
                           for path in corpus_files}}
# Save the reviewed benchmark once and reuse it for every retrieval method.
MANIFEST_PATH = DATA_DIR / "answer_spans.json"

def norm(text):
    return " ".join(text.lower().split())

# Preserve document wording so proposed spans can be checked against their source.
CORPUS_TEXT = {path.name: path.read_text(encoding="utf-8") for path in corpus_files}
# Ignore case and whitespace when checking whether a span occurs more than once.
CORPUS_NORM = norm(" ".join(CORPUS_TEXT.values()))
# Common words should not dominate the answer-overlap score.
STOP = set("the a an of and or to in on for with by is are was were be as at from that this it its which".split())

def content_words(text):
    return set(re.findall(r"[a-z0-9]+", text.lower())) - STOP

if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    # Replace a saved benchmark whose inputs have changed.
    if not isinstance(manifest, dict) or manifest.get("input_hashes") != INPUT_HASHES:
        MANIFEST_PATH.unlink()
        print("Saved benchmark did not match the current inputs; replacing it.")

if MANIFEST_PATH.exists():
    # Reuse the saved benchmark instead of selecting a new one.
    QUERIES = manifest["queries"]
    print("Loaded the saved benchmark.")
else:
    # Keep candidate spans short enough to review, with their source filename attached.
    sentences = [(name, sentence) for name, text in CORPUS_TEXT.items()
                 for sentence in re.split(r"(?<=[.!?])\s+", " ".join(text.split()))
                 if 8 <= len(sentence.split()) <= 40]
    if not sentences:
        raise ValueError("No candidate sentences found; annotate answer spans manually.")
    # Overlap ranks proposals; human review still decides whether they answer the question.
    candidates = []
    for row in eval_rows:
        # Search using the informative words from the reference answer.
        wanted = content_words(row["response"])
        # Keep both the matching sentence and its source for the review table.
        source, span = max(sentences, key=lambda pair: len(wanted & content_words(pair[1])) /
                          max(len(content_words(pair[1])), 1) ** 0.5)
        if CORPUS_NORM.count(norm(span)) == 1:
            candidates.append({"query": row["instruction"], "expected_answer": row["response"],
                               "source_file": source, "span": span,
                               "overlap": len(wanted & content_words(span))})
    # Choose ten distinct questions with distinct spans; more than one may come from the same document.
    selected, seen_queries, seen_spans = [], set(), set()
    for candidate in sorted(candidates, key=lambda item: -item["overlap"]):
        if candidate["query"] not in seen_queries and norm(candidate["span"]) not in seen_spans:
            selected.append(candidate)
            seen_queries.add(candidate["query"])
            seen_spans.add(norm(candidate["span"]))
        if len(selected) == 10:
            break
    if len(selected) != 10:
        raise ValueError("Could not propose ten distinct queries; select and annotate ten evaluation queries manually.")
    # Stable IDs link the retrieval results, generated answers, and judge verdicts.
    QUERIES = [{"qid": f"Q{number}", **{key: value for key, value in item.items() if key != "overlap"}}
               for number, item in enumerate(selected, 1)]
print(f"Evaluation rows: {len(eval_rows)}; selected queries: {len(QUERIES)}")

Loaded the saved benchmark.
Evaluation rows: 80; selected queries: 10


### Review the Spans

Read the candidate table, then fill `SPAN_REVIEW` using each exact question as its key. Each entry needs `approved: True`, `source_file`, and a verbatim `span` that answers the question. Keep the Assignment 1 reference unchanged. All ten annotations must be approved before retrieval scoring.


In [6]:
# Before you start: review the answer spans and check the ten query/reference pairs.
# Key entries by exact question text; approve only after reading the source and reference.
SPAN_REVIEW = {
    # Q1
    "What should be considered when planning detection of undiagnosed T2DM and prevention programmes in low- and middle-income countries?": {
        "source_file": "idf-clinical-practice-recommendations-2025.txt",
        "span": "GLOBAL CONSIDERATIONS\nThe capacity to detect individuals with\nundiagnosed T2DM and IH and implement\ndiabetes prevention programmes varies\nconsiderably between countries and health\nservices and systems, especially in low- and\nmiddle-income countries (LMIC).",
        "approved": True,
    },
    # Q2
    "Which foods should people with diabetes and dyslipidaemia restrict because of their saturated and trans fats?": {
        "source_file": "icmr-type2-diabetes-guidelines-2018.txt",
        "span": "(b) Dyslipidaemia : Saturated and trans fats food sources like vanaspati, butter,\nghee, margarine, coconut oil, red meats like sausages, ham, bacon, egg yellow, whole\nmilk and its products should be restricted.",
        "approved": True,
    },
    # Q3
    "How much flax seed can vegetarians take daily to obtain omega-3 fatty acids?": {
        "source_file": "icmr-type2-diabetes-guidelines-2018.txt",
        "span": "Vegetarians can take flax seeds (10 g / day) in their diet as both fish\nand flax seeds are rich in omega-3 fatty acids which is protective for heart disease.",
        "approved": True,
    },
    # Q4
    "Why does WHO conduct a periodic survey of country capacity for noncommunicable diseases?": {
        "source_file": "idf-clinical-practice-recommendations-2025.txt",
        "span": "The World\nHealth Organization (WHO) periodically\nconducts a country capacity survey on noncommunicable diseases (NCDs) to assess the\nnational-level response to the NCD burden,\nwhich includes information on health system\ncapacity for diabetes detection, treatment, and\ncare.",
        "approved": True,
    },
    # Q5
    "How should glycaemic targets be set for people with T2DM?": {
        "source_file": "moh-malaysia-type2-diabetes-cpg-2020.txt",
        "span": "Therapy\nin most patients with T2DM should be targeted to\nachieve an HbA1c\u22647.0% or 6.5% (where appropriate),\nif achievable without significant hypoglycaemia.",
        "approved": True,
    },
    # Q6
    "How many data sources underpinned the IDF diabetes prevalence estimates, and what period did the research cover?": {
        "source_file": "idf-atlas-global-diabetes-prevalence-2021.txt",
        "span": "Methods:\u00a0A total of 219 data sources meeting pre-established quality criteria reporting\nresearch conducted between 2005 and 2020 and representing 215 countries and territories were\nidentified.",
        "approved": True,
    },
    # Q7
    "How should people starting a sulphonylurea be counselled about hypoglycaemia warning signs?": {
        "source_file": "sign154-pharmacological-glycaemic-control.txt",
        "span": "The warning signs of hypoglycaemia, which should be outlined to people taking these agents, include (early\nsigns) tremor, sweating, shaking, irritability, and (later signs) lack of concentration and co-ordination.",
        "approved": True,
    },
    # Q8
    "How should the diet be changed during fever or other illness in diabetes?": {
        "source_file": "icmr-type2-diabetes-guidelines-2018.txt",
        "span": "6.2.3 Special situations requiring dietary modification:\n(i) Sick Days\nIn the event of fever or other illness, the diabetic diet should be modified by\nchanging the consistency and texture of foods to maintain adequate calorie intake.",
        "approved": True,
    },
    # Q9
    "For how many days is each type of continuous glucose monitoring used when advanced monitoring is needed?": {
        "source_file": "icmr-type2-diabetes-guidelines-2018.txt",
        "span": "Here\nContinuous Glucose Monitoring (CGM) for 5-7 days or iSCGM (intermittently scanned\nCGM) for 14 days may be used.",
        "approved": True,
    },
    # Q10
    "Which foods and fluids should be encouraged for people with diabetes on sick days?": {
        "source_file": "icmr-type2-diabetes-guidelines-2018.txt",
        "span": "Semi solid foods and fluids or items like thin soups, milk, buttermilk, or fresh lime juice\nshould be encouraged.",
        "approved": True,
    },
}

if not MANIFEST_PATH.exists():
    show_md_table(["Query", "Expected answer", "Candidate source", "Candidate span"],
                  [(item["query"], item["expected_answer"], item["source_file"], item["span"])
                   for item in QUERIES], title="Candidates to review before retrieval scoring")
    pending = []
    for item in QUERIES:
        review = SPAN_REVIEW.get(item["query"], {})
        if review.get("approved") is not True:
            pending.append(item["qid"])
            continue
        item["source_file"] = review["source_file"]
        item["span"] = review["span"]
        item["human_reviewed"] = True
    # Candidate overlap is not proof that a span answers its question; require human approval.
    if pending:
        raise ValueError(f"Review SPAN_REVIEW for {', '.join(pending)}, then rerun this cell.")

# Preserve the original evaluation references and verify each span against its named document.
expected_pairs = {(row["instruction"], row["response"]) for row in eval_rows}
assert len(QUERIES) == 10
assert len({item["query"] for item in QUERIES}) == 10
assert [item["qid"] for item in QUERIES] == [f"Q{number}" for number in range(1, 11)]
for item in QUERIES:
    assert item.get("human_reviewed") is True, f"{item['qid']}: annotation is not reviewed"
    assert (item["query"], item["expected_answer"]) in expected_pairs, "Query/reference is not from the evaluation split"
    source = CORPUS_TEXT.get(item["source_file"], "")
    assert item["span"].strip() and norm(item["span"]) in norm(source), "Span is absent from its named source"
print("All ten query/reference pairs and human-reviewed source spans validated.")

All ten query/reference pairs and human-reviewed source spans validated.


In [7]:
# Before you start: save the reviewed benchmark and define the hit test.
def is_hit(chunk_text, span):
    # A hit is a normalized literal match of the span inside the chunk.
    return norm(span) in norm(chunk_text)

# Save the reviewed benchmark with its input hashes.
manifest = {"input_hashes": INPUT_HASHES, "queries": QUERIES}
if not MANIFEST_PATH.exists():
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
show_md_table(["qid", "source file", "query", "expected answer", "answer span"],
              [(item["qid"], item["source_file"], item["query"], item["expected_answer"], item["span"])
               for item in QUERIES], title="Evaluation benchmark")

**Evaluation benchmark**

| qid | source file | query | expected answer | answer span |
| --- | --- | --- | --- | --- |
| Q1 | idf-clinical-practice-recommendations-2025.txt | What should be considered when planning detection of undiagnosed T2DM and prevention programmes in low- and middle-income countries? | Planners should expect that capacity to detect undiagnosed T2DM and intermediate hyperglycaemia and to implement diabetes prevention programmes varies considerably between countries, health services and systems, and the variation is especially pronounced in low- and middle-income countries. | GLOBAL CONSIDERATIONS The capacity to detect individuals with undiagnosed T2DM and IH and implement diabetes prevention programmes varies considerably between countries and health services and systems, especially in low- and middle-income countries (LMIC). |
| Q2 | icmr-type2-diabetes-guidelines-2018.txt | Which foods should people with diabetes and dyslipidaemia restrict because of their saturated and trans fats? | Restrict vanaspati, butter, ghee, margarine, coconut oil, red meats such as sausages, ham and bacon, egg yellow, and whole milk along with its products, since these are saturated and trans fat sources. | (b) Dyslipidaemia : Saturated and trans fats food sources like vanaspati, butter, ghee, margarine, coconut oil, red meats like sausages, ham, bacon, egg yellow, whole milk and its products should be restricted. |
| Q3 | icmr-type2-diabetes-guidelines-2018.txt | How much flax seed can vegetarians take daily to obtain omega-3 fatty acids? | Vegetarians can include 10 g of flax seeds per day in their diet, which supplies omega-3 fatty acids that protect against heart disease, in the same way that fish does. | Vegetarians can take flax seeds (10 g / day) in their diet as both fish and flax seeds are rich in omega-3 fatty acids which is protective for heart disease. |
| Q4 | idf-clinical-practice-recommendations-2025.txt | Why does WHO conduct a periodic survey of country capacity for noncommunicable diseases? | WHO conducts the survey to assess how each country is responding nationally to the NCD burden. It gathers information on health system capacity for diabetes detection, treatment, and care, which helps show the differences between countries and systems. | The World Health Organization (WHO) periodically conducts a country capacity survey on noncommunicable diseases (NCDs) to assess the national-level response to the NCD burden, which includes information on health system capacity for diabetes detection, treatment, and care. |
| Q5 | moh-malaysia-type2-diabetes-cpg-2020.txt | How should glycaemic targets be set for people with T2DM? | Glycaemic targets must be individualised for each patient. Therapy in most patients should aim for an HbA1c of 7.0% or lower, or 6.5% where appropriate, as long as this is achievable without significant hypoglycaemia occurring. | Therapy in most patients with T2DM should be targeted to achieve an HbA1c≤7.0% or 6.5% (where appropriate), if achievable without significant hypoglycaemia. |
| Q6 | idf-atlas-global-diabetes-prevalence-2021.txt | How many data sources underpinned the IDF diabetes prevalence estimates, and what period did the research cover? | Researchers used 219 data sources that satisfied pre-established quality criteria, covering studies done in 2005-2020 and representing 215 countries and territories, as the basis for the IDF prevalence estimates. | Methods: A total of 219 data sources meeting pre-established quality criteria reporting research conducted between 2005 and 2020 and representing 215 countries and territories were identified. |
| Q7 | sign154-pharmacological-glycaemic-control.txt | How should people starting a sulphonylurea be counselled about hypoglycaemia warning signs? | They should be told that early warning signs include tremor, sweating, shaking and irritability, while later signs involve poor concentration and impaired co-ordination. Recognising both stages lets them act before the episode becomes severe. | The warning signs of hypoglycaemia, which should be outlined to people taking these agents, include (early signs) tremor, sweating, shaking, irritability, and (later signs) lack of concentration and co-ordination. |
| Q8 | icmr-type2-diabetes-guidelines-2018.txt | How should the diet be changed during fever or other illness in diabetes? | The consistency and texture of the diabetic diet should be modified on sick days, so that the person is still able to maintain adequate calorie intake while they are unwell with fever. | 6.2.3 Special situations requiring dietary modification: (i) Sick Days In the event of fever or other illness, the diabetic diet should be modified by changing the consistency and texture of foods to maintain adequate calorie intake. |
| Q9 | icmr-type2-diabetes-guidelines-2018.txt | For how many days is each type of continuous glucose monitoring used when advanced monitoring is needed? | Continuous Glucose Monitoring (CGM) is applied for 5-7 days, whereas intermittently scanned CGM (iSCGM) is used for a longer period of 14 days, whenever advanced monitoring is required for a patient. | Here Continuous Glucose Monitoring (CGM) for 5-7 days or iSCGM (intermittently scanned CGM) for 14 days may be used. |
| Q10 | icmr-type2-diabetes-guidelines-2018.txt | Which foods and fluids should be encouraged for people with diabetes on sick days? | Semi solid foods and fluids are encouraged, with thin soups, milk, buttermilk and fresh lime juice given as examples, because these help the person keep taking in calories while ill. | Semi solid foods and fluids or items like thin soups, milk, buttermilk, or fresh lime juice should be encouraged. |

## Part A - Dense Search and Chunk Size (5 marks)

Before you start: for each of your 10 queries, mark the answer span, the sentence or short passage in your corpus that answers it. A retrieved chunk is a hit if it contains that span. You will reuse these spans in Parts A, B and C. Also note your embedding model's maximum input length, for example 256 tokens for all-MiniLM-L6-v2, and count tokens with that model's tokenizer.

In [8]:
# A1 setup: load the embedding model that A1.1 and A1.2 use.
from sentence_transformers import SentenceTransformer
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

EMBED_MODEL_ID = "BAAI/bge-base-en-v1.5"
EMBED_BATCH = 64

with working("Loading the embedding model"):
    embed_model = SentenceTransformer(EMBED_MODEL_ID, device=L.DEVICE)
embed_tok = embed_model.tokenizer
EMBED_DIM = embed_model.get_sentence_embedding_dimension()
EMBED_MAX_TOKENS = embed_model.max_seq_length
print(f"Embedding model: {EMBED_MODEL_ID}; maximum input length: {EMBED_MAX_TOKENS} tokens")

Loading the embedding model...
Loading the embedding model: completed in 20.5s
Embedding model: BAAI/bge-base-en-v1.5; maximum input length: 512 tokens


### A1.1 Dense search function (1 mark)

One function embeds the chunks, builds a FAISS `IndexFlatIP` index and returns the top k chunks for each query. Embeddings are L2-normalised, so the inner product equals cosine similarity. The index is cached per chunk set, so Parts B and C reuse it.


In [9]:
# A1.1: reusable dense search function. It embeds the chunks, builds a FAISS IndexFlatIP index and returns the top k chunks for a query.
import faiss

# Cache each chunk set so repeated searches do not rebuild its embeddings and index.
_INDEX_CACHE = {}

print("Dense search helpers ready")

# Demo: the top 3 chunks for the first query, using 100-token chunks.
demo_hits = dense_search(chunk_corpus(100), [QUERIES[0]["query"]], 3)[0]
print(f"Top 3 chunks for {QUERIES[0]['qid']}: {QUERIES[0]['query']}")
for hit in demo_hits:
    print(f"[{hit['rank']}] {hit['chunk_id']} (score {hit['score']:.3f}): {hit['text'][:160].replace(chr(10), ' ')}")


Dense search helpers ready
Top 3 chunks for Q1: What should be considered when planning detection of undiagnosed T2DM and prevention programmes in low- and middle-income countries?
[1] idf-clinical-practice-recommendations-2025#53 (score 0.838): oglobin (HbA1c). Programmes for the detection of undiagnosed T2DM and the prevention of progression to T2DM in highrisk individuals are well-accepted strategies
[2] idf-clinical-practice-recommendations-2025#176 (score 0.783): . Catley D, Puoane T, Goggin K, et al. Adapting the Diabetes Prevention Program for lowand middle-income countries: preliminary implementation findings from lif
[3] idf-clinical-practice-recommendations-2025#12 (score 0.780): support, T2DM can be controlled and its associated complications prevented or delayed. Furthermore, there is increasing evidence that remission of T2DM may be p


### A1.2 Dense search setup (1 mark)

Report the embedding model name, vector size, similarity measure, batch size, and how long the index takes to build for your final chunk set. (1 mark)

In [10]:
# A1.2 setup: the two chunk sizes the brief tests. Each size is chunked with chunk_corpus(size).
CHUNK_SIZES = [100, 400]

print(f"Chunk sizes tested for A1.2 and A2.1: {CHUNK_SIZES}")


Chunk sizes tested for A1.2 and A2.1: [100, 400]


In [11]:
# A1.2: report the embedding model, vector size, similarity, batch size and index build time for each chunk size.

print_table(["Setting", "Value"], [
    ("Embedding model", EMBED_MODEL_ID),
    ("Vector size", str(EMBED_DIM)),
    ("Similarity measure", "inner product on L2-normalised vectors (cosine)"),
    ("Batch size", str(EMBED_BATCH)),
], title="A1.2 dense search setup")

a12_rows = []
for size in CHUNK_SIZES:
    chunks_for_size = chunk_corpus(size)
    a12_rows.append((str(size), f"{len(chunks_for_size):,}", f"{index_build_seconds(chunks_for_size):.2f}"))
print_table(["Chunk size (tokens)", "Total chunks", "Index build time (s)"], a12_rows,
            title="A1.2 index build time for each chunk size")


A1.2 dense search setup
Setting                                                        Value
--------------------------------------------------------------------
Embedding model                                BAAI/bge-base-en-v1.5
Vector size                                                      768
Similarity measure   inner product on L2-normalised vectors (cosine)
Batch size                                                        64
A1.2 index build time for each chunk size
Chunk size (tokens)   Total chunks   Index build time (s)
---------------------------------------------------------
100                          6,663                   8.16
400                          1,672                   7.96


### A2.1 Chunk size test (2 marks)

Fixed-size chunks with no overlap, per document, counted with the embedding tokenizer. The text of each chunk is sliced from the original document with the tokenizer's offsets, so the generator sees the original wording.

| Column | How |
| --- | --- |
| Total chunks | number of chunks over all corpus documents |
| Hit rate@3 (%) | share of the 10 queries where one of the top 3 chunks contains the span |
| Tokens sent per query (top 3) | combined `n_tokens` of the top 3, averaged over the 10 queries |

In [12]:
# A2.1: chunk at 100 and 400 tokens and fill the hit-rate table.
a2_hit = {}
a2_tokens = {}
chunk_sets = {}
a2_rows = []

for size in CHUNK_SIZES:
    chunks_for_size = chunk_corpus(size)
    assert max(chunk["n_tokens"] for chunk in chunks_for_size) <= size
    chunk_sets[size] = chunks_for_size
    chunks_by_id = {chunk["chunk_id"]: chunk for chunk in chunks_for_size}
    top_results = dense_search(chunks_for_size, [item["query"] for item in QUERIES], 3)

    # A query counts as a hit if any of its top three chunks contains the answer span.
    hits = sum(
        any(is_hit(result["text"], item["span"]) for result in ranked_chunks)
        for item, ranked_chunks in zip(QUERIES, top_results)
    )
    # Token cost is the combined length of the three chunks, averaged across queries.
    mean_context_tokens = sum(
        sum(chunks_by_id[result["chunk_id"]]["n_tokens"] for result in ranked_chunks)
        for ranked_chunks in top_results
    ) / len(QUERIES)

    a2_hit[size] = 100 * hits / len(QUERIES)
    a2_tokens[size] = mean_context_tokens
    a2_rows.append((
        str(size), f"{len(chunks_for_size):,}",
        f"{a2_hit[size]:.0f}", f"{mean_context_tokens:.0f}",
    ))

print_table(
    ["Chunk size (tokens)", "Total chunks", "Hit rate@3 (%)", "Tokens sent per query (top 3)"],
    a2_rows, title="A2 chunk size test (top 3, 10 queries)",
)
too_long = [size for size in CHUNK_SIZES if size > EMBED_MAX_TOKENS]
print(f"Chunk sizes over the {EMBED_MAX_TOKENS}-token embedding limit: {too_long}" if too_long else
      f"Both chunk sizes fit the {EMBED_MAX_TOKENS}-token embedding limit, so no chunk text is cut off.")

A2 chunk size test (top 3, 10 queries)
Chunk size (tokens)   Total chunks   Hit rate@3 (%)   Tokens sent per query (top 3)
-----------------------------------------------------------------------------------
100                          6,663               50                             300
400                          1,672               20                            1200
Both chunk sizes fit the 512-token embedding limit, so no chunk text is cut off.


In [13]:
# Shared setup for A2.2 and D1.1: the grounding prompt and the builder that fills it.
GEN_PROMPT = """Answer the question using ONLY the numbered chunks below.
Answer in full sentences. Cite the number of each chunk you used, for example [2], using only the numbers of the chunks below.
If the chunks do not contain the answer, reply exactly: Not found in the documents

Chunks:
[1] {chunk_1}
[2] {chunk_2}
[3] {chunk_3}

Question: {question}"""

print("Grounding prompt and builder ready; the D1.1 cell prints them.")


Grounding prompt and builder ready; the D1.1 cell prints them.


### A2.2 Chosen chunk size (1 mark)

In two or three sentences, name the chunk size you will use in Parts B to D and why. (1 mark)

In [14]:
# A2.2: choose the chunk size whose complete generation prompts fit the context limit.
from transformers import AutoConfig, AutoTokenizer

GEN_MAX_NEW_TOKENS = 256
# The adapter's training length is not the model's context limit.
ADAPTER_TRAINING_LENGTH = 512
mistral_tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)
generator_config = AutoConfig.from_pretrained(CPT_DIR)
# Some tokenizers store a huge placeholder when there is no limit; ignore it.
context_limits = [int(generator_config.max_position_embeddings)]
if 0 < mistral_tok.model_max_length < 1_000_000:
    context_limits.append(int(mistral_tok.model_max_length))
GEN_CONTEXT_LIMIT = min(context_limits)
PROMPT_LENGTHS = {}
for size in CHUNK_SIZES:
    retrieved = dense_search(chunk_sets[size], [item["query"] for item in QUERIES], 3)
    # Include instructions, question, chunk labels, and the actual chat wrapper.
    PROMPT_LENGTHS[size] = [len(mistral_tok(build_generator_prompt(mistral_tok, item["query"],
                            [chunk["text"] for chunk in top]), add_special_tokens=False).input_ids)
                            for item, top in zip(QUERIES, retrieved)]
# Every query must fit while leaving room for the generated answer.
feasible_sizes = [size for size in CHUNK_SIZES
                  if max(PROMPT_LENGTHS[size]) + GEN_MAX_NEW_TOKENS <= GEN_CONTEXT_LIMIT]
if not feasible_sizes:
    raise ValueError("Neither chunk size fits all complete prompts plus the answer budget; revise the context policy.")
# Prefer hit rate, then lower retrieved-token cost, then the smaller chunk size.
CHUNK_SIZE = min(feasible_sizes, key=lambda size: (-a2_hit[size], a2_tokens[size], size))
chunks = chunk_sets[CHUNK_SIZE]
print_table(["Chunk size", "Minimum prompt tokens", "Maximum prompt tokens", "Output budget", "Context limit"],
            [(str(size), str(min(PROMPT_LENGTHS[size])), str(max(PROMPT_LENGTHS[size])),
              str(GEN_MAX_NEW_TOKENS), str(GEN_CONTEXT_LIMIT)) for size in CHUNK_SIZES],
            title="Complete dense top-3 prompt budgets for all ten queries")
print(f"Chosen chunk size: {CHUNK_SIZE}; adapter training length: {ADAPTER_TRAINING_LENGTH}")

Complete dense top-3 prompt budgets for all ten queries
Chunk size   Minimum prompt tokens   Maximum prompt tokens   Output budget   Context limit
------------------------------------------------------------------------------------------
100                            454                     556             256           32768
400                           1504                    1756             256           32768
Chosen chunk size: 100; adapter training length: 512


In [15]:
# A2.2: name the chunk size used in Parts B to D and why.
from IPython.display import Markdown, display

longer = sum(length > ADAPTER_TRAINING_LENGTH for length in PROMPT_LENGTHS[CHUNK_SIZE])
training_note = (f"{longer} of 10 prompt{'' if longer == 1 else 's'} {'is' if longer == 1 else 'are'} longer than the adapter's {ADAPTER_TRAINING_LENGTH}-token training length, so check answer quality on those."
                 if max(PROMPT_LENGTHS[CHUNK_SIZE]) > ADAPTER_TRAINING_LENGTH else
                 f"All prompts are within the adapter's {ADAPTER_TRAINING_LENGTH}-token training length.")
display(Markdown(
    f"**A2.2: Use {CHUNK_SIZE}-token chunks in Parts B-D.** Hit rate@3 was {a2_hit[100]:.0f}% at 100 tokens "
    f"and {a2_hit[400]:.0f}% at 400 tokens, with {a2_tokens[100]:.0f} and {a2_tokens[400]:.0f} retrieved tokens per query. "
    f"The higher hit rate decides; ties go to fewer tokens. "
    f"The longest complete prompt is {max(PROMPT_LENGTHS[CHUNK_SIZE])} tokens plus {GEN_MAX_NEW_TOKENS} answer tokens, "
    f"within the {GEN_CONTEXT_LIMIT}-token limit. {training_note}"))

**A2.2: Use 100-token chunks in Parts B-D.** Hit rate@3 was 50% at 100 tokens and 20% at 400 tokens, with 300 and 1200 retrieved tokens per query. The higher hit rate decides; ties go to fewer tokens. The longest complete prompt is 556 tokens plus 256 answer tokens, within the 32768-token limit. 2 of 10 prompts are longer than the adapter's 512-token training length, so check answer quality on those.

## Part B - Search Comparison (4 marks)

Use the chosen chunks and the same ten queries. Compare Dense, BM25, and Hybrid at top five. Hybrid fuses the top 20 from each method using `1 / (60 + rank)` per list; ranks start at one and missing entries contribute zero.

Report mean per-query latency after one warm-up, hit rate@1, and hit rate@3. A hit requires the annotated answer span.


In [16]:
# Part B setup: shared evaluation for hit rates and query latency.

print("Evaluation helper ready")


Evaluation helper ready


### B1 Dense (1 mark)


In [17]:
# B1: dense search on the ten queries.
b_results = {}

b_results["Dense"] = evaluate(dense_fn)
b1_row = b_results["Dense"]
print_table(["Method", "Average query latency (ms)", "Hit rate@1 (%)", "Hit rate@3 (%)"],
            [("Dense", f"{b1_row['latency_ms']:.1f}", f"{b1_row['hit1']:.0f}", f"{b1_row['hit3']:.0f}")],
            title="Part B1: Dense row (top 5)")


Part B1: Dense row (top 5)
Method   Average query latency (ms)   Hit rate@1 (%)   Hit rate@3 (%)
---------------------------------------------------------------------
Dense                           8.5               30               50


### B2 BM25 (1 mark)


In [18]:
# B2: BM25 search on the same chunks.
from rank_bm25 import BM25Okapi

# Index the same chunks the dense search uses.
bm25 = BM25Okapi([bm25_tokens(chunk["text"]) for chunk in chunks])

b_results["BM25"] = evaluate(bm25_fn)
b2_row = b_results["BM25"]
print_table(["Method", "Average query latency (ms)", "Hit rate@1 (%)", "Hit rate@3 (%)"],
            [("BM25", f"{b2_row['latency_ms']:.1f}", f"{b2_row['hit1']:.0f}", f"{b2_row['hit3']:.0f}")],
            title="Part B2: BM25 row (top 5)")


Part B2: BM25 row (top 5)
Method   Average query latency (ms)   Hit rate@1 (%)   Hit rate@3 (%)
---------------------------------------------------------------------
BM25                           12.0               50               50


### B3 Hybrid with RRF (1 mark)


In [19]:
# B3: hybrid search with reciprocal rank fusion.

chunk_by_id = {chunk["chunk_id"]: chunk for chunk in chunks}

b_results["Hybrid (RRF)"] = evaluate(hybrid_fn)
b3_row = b_results["Hybrid (RRF)"]
print_table(["Method", "Average query latency (ms)", "Hit rate@1 (%)", "Hit rate@3 (%)"],
            [("Hybrid (RRF)", f"{b3_row['latency_ms']:.1f}", f"{b3_row['hit1']:.0f}", f"{b3_row['hit3']:.0f}")],
            title="Part B3: Hybrid (RRF) row (top 5)")


Part B3: Hybrid (RRF) row (top 5)
Method         Average query latency (ms)   Hit rate@1 (%)   Hit rate@3 (%)
---------------------------------------------------------------------------
Hybrid (RRF)                         24.3               50               60


In [20]:
# B summary: the three methods side by side.

print_table(
    ["Method", "Average query latency (ms)", "Hit rate@1 (%)", "Hit rate@3 (%)"],
    [
        (method, f"{metrics['latency_ms']:.1f}", f"{metrics['hit1']:.0f}", f"{metrics['hit3']:.0f}")
        for method, metrics in b_results.items()
    ],
    title=f"Part B search comparison (chunk size {CHUNK_SIZE}, top 5)",
)


Part B search comparison (chunk size 100, top 5)
Method         Average query latency (ms)   Hit rate@1 (%)   Hit rate@3 (%)
---------------------------------------------------------------------------
Dense                                 8.5               30               50
BM25                                 12.0               50               50
Hybrid (RRF)                         24.3               50               60


### B4 Best method (1 mark)


In [21]:
# B4: choose the best method and show a ranking example.
BEST_METHOD = max(b_results, key=lambda method: (b_results[method]["hit1"], b_results[method]["hit3"],
                                                -b_results[method]["latency_ms"]))

example = next((item for item in QUERIES
                if answer_rank(BEST_METHOD, item) < min(answer_rank(method, item)
                   for method in b_results if method != BEST_METHOD)), None)
if example is None:
    example = next((item for item in QUERIES
                    if len({answer_rank(method, item) for method in b_results}) > 1), QUERIES[0])
best = b_results[BEST_METHOD]
tied = [method for method in b_results if method != BEST_METHOD and
        (b_results[method]["hit1"], b_results[method]["hit3"]) == (best["hit1"], best["hit3"])]
choice_reason = (f"Its hit rates tie with {', '.join(tied)}, so lower measured latency breaks the tie."
                 if tied else "It wins by hit rate@1, followed by hit rate@3.")
rank_description = "; ".join(f"{method}: rank {rank_of(method, example['qid'], example['span'])}"
                              for method in b_results)
display(Markdown(
    f"**B4: {BEST_METHOD} is selected**, with {best['hit1']:.0f}% hit rate@1, {best['hit3']:.0f}% "
    f"hit rate@3 and {best['latency_ms']:.1f} ms/query. {choice_reason} "
    f"For {example['qid']} ('{example['query']}'), the annotated answer span ranks as follows: "
    f"{rank_description} ('-' means absent from the top five). "
    "These results cover only these ten queries on this hardware."))
example_rows = []
for method, result in b_results.items():
    for chunk in result["ranked"][example["qid"]]:
        example_rows.append((method, str(chunk["rank"]), str(is_hit(chunk["text"], example["span"])),
                             chunk["chunk_id"], chunk["text"]))
show_md_table(["Method", "Rank", "Span hit", "Chunk", "Text"], example_rows,
              title=f"B4 evidence: {example['qid']}; answer span: {example['span']}")


**B4: Hybrid (RRF) is selected**, with 50% hit rate@1, 60% hit rate@3 and 24.3 ms/query. It wins by hit rate@1, followed by hit rate@3. For Q5 ('How should glycaemic targets be set for people with T2DM?'), the annotated answer span ranks as follows: Dense: rank -; BM25: rank -; Hybrid (RRF): rank 2 ('-' means absent from the top five). These results cover only these ten queries on this hardware.

**B4 evidence: Q5; answer span: Therapy
in most patients with T2DM should be targeted to
achieve an HbA1c≤7.0% or 6.5% (where appropriate),
if achievable without significant hypoglycaemia.**

| Method | Rank | Span hit | Chunk | Text |
| --- | --- | --- | --- | --- |
| Dense | 1 | False | sign154-pharmacological-glycaemic-control#138 | group as the key clinical recommendations that should be prioritised for implementation. 2.1 TARGETS FOR GLYCAEMIC CONTROL R An HbA1c target of 7.0% (53 mmol/mol) among people with type 2 diabetes is reasonable to reduce the risk of microvascular and macrovascular disease. A target of 6.5% (48 mmol/mol) may be appropriate at diagnosis. Targets should be set with individuals in order to balance benefits |
| Dense | 2 | False | idf-clinical-practice-recommendations-2025#213 | 5%–7.1% (48–54 mmol/mol).4-8 These data are the basis for the general target of HbA1c <7.0% (<53 mmol/mol) in people with T2DM recommended by most guidelines for minimising risk of diabetes complications and hypoglycaemia. This target is proposed as a guide, which is appropriate for many people, but it is important to consider personalising targets, |
| Dense | 3 | False | racgp-gp-management-type2-diabetes-2016#897 | . 132. Cheung NW, Conn JJ, d’Emden MC, et al. Position statement of the Australian Diabetes Society: Individualisation of glycated haemoglobin targets for adults with diabetes mellitus. Med J Aust 2009;191(6):339–44. 133. Ahren B. Avoiding hypoglycemia: A key to success for glucose-lowering therapy in type 2 diabetes. Vasc Health Risk Manag 2013;9: |
| Dense | 4 | False | waikato-gp-type2-diabetes-manual-2024#136 | tion, healthy diet, regular physical activity, optimal weight) in everyone. • Optimise glycaemic control to an appropriate level in consultation with the individual patient. Agreed target range will generally be more stringent in younger and fitter patients (e.g. HbA1c 48-53, FBG 5-7mmol/L without hypos) than older, co-morbid or frail patients and those prone to hypoglycaemia ( |
| Dense | 5 | False | idf-clinical-practice-recommendations-2025#181 | GLYCAEMIC TARGETS o Glycaemic control reduces the risk of both micro- and macrovascular complications in type 2 diabetes mellitus (T2DM). o Assessing and monitoring glycaemia is an essential component of guiding treatment decisions to achieve and maintain target glycaemic control. o Setting glycaemic targets is based on established associations with adverse outcomes. o HbA1c is the established gold standard for assessing glycaemic |
| BM25 | 1 | False | sign154-pharmacological-glycaemic-control#138 | group as the key clinical recommendations that should be prioritised for implementation. 2.1 TARGETS FOR GLYCAEMIC CONTROL R An HbA1c target of 7.0% (53 mmol/mol) among people with type 2 diabetes is reasonable to reduce the risk of microvascular and macrovascular disease. A target of 6.5% (48 mmol/mol) may be appropriate at diagnosis. Targets should be set with individuals in order to balance benefits |
| BM25 | 2 | False | sign154-pharmacological-glycaemic-control#163 | diabetes is reasonable to reduce the risk of microvascular and macrovascular disease. A target of 6.5% (48 mmol/mol) may be appropriate at diagnosis. Targets should be set with individuals in order to balance benefits with harms, in particular hypoglycaemia and weight gain. 3 • Targets for glycaemic control 10 / 4 Metformin Metformin is a small molecule from the biguanide family that has been used as a |
| BM25 | 3 | False | idf-clinical-practice-recommendations-2025#235 | ing with CGM data, Table 2.1 shows suggested general targets for SMBG glucose levels. Algorithms based on similar targets have been used to safely and successfully adjust basal insulin in people with T2DM.57 Glycaemic profiles / other measures of glycaemia While measurement of HbA1c is the established gold standard biomarker for assessing glycaemic control, it does not provide information on shortterm glycaemic variability or hyp |
| BM25 | 4 | False | moh-malaysia-type2-diabetes-cpg-2020#669 | CHRONIC COMPLICATIONS Recommendations: Management of CVD 2. Multi-factorial management to achieve glycaemic, BP and lipid targets to reduce CV events. Grade A 3. Aspirin should be used for secondary prevention in T2DM patients. For patients with ASCVD and documented aspirin allergy, clopidogrel (75 mg/day) should be used. Grade A 4. Primary prevention of ASCVD with low dose aspirin (100 mg |
| BM25 | 5 | False | moh-malaysia-type2-diabetes-cpg-2020#142 | of the legacy effects/metabolic memory of good glucose control should be emphasized to all newly diagnosed diabetic patients. Grade A MANAGEMENT OF TYPE 2 DIABETES MELLITUS 3.4 Targets for control SUMMARY OF UPDATES • The importance of achieving optimal body weight with lifestyle modification is emphasized in overweight/obese T2DM individuals. • HbA 1c targets: glycaemic targets have been revised according to individual characteristics. • BP targets are revised to 130-139/70-79 |
| Hybrid (RRF) | 1 | False | sign154-pharmacological-glycaemic-control#138 | group as the key clinical recommendations that should be prioritised for implementation. 2.1 TARGETS FOR GLYCAEMIC CONTROL R An HbA1c target of 7.0% (53 mmol/mol) among people with type 2 diabetes is reasonable to reduce the risk of microvascular and macrovascular disease. A target of 6.5% (48 mmol/mol) may be appropriate at diagnosis. Targets should be set with individuals in order to balance benefits |
| Hybrid (RRF) | 2 | True | moh-malaysia-type2-diabetes-cpg-2020#392 | emic control, co-morbidities, complications and other CVD risk factors should be done at initial visit and whenever indicated subsequently. Grade C 1. Glycaemic targets must be individualised. Therapy in most patients with T2DM should be targeted to achieve an HbA1c≤7.0% or 6.5% (where appropriate), if achievable without significant hypoglycaemia. Reduction in HbA1c has been shown to |
| Hybrid (RRF) | 3 | False | idf-clinical-practice-recommendations-2025#181 | GLYCAEMIC TARGETS o Glycaemic control reduces the risk of both micro- and macrovascular complications in type 2 diabetes mellitus (T2DM). o Assessing and monitoring glycaemia is an essential component of guiding treatment decisions to achieve and maintain target glycaemic control. o Setting glycaemic targets is based on established associations with adverse outcomes. o HbA1c is the established gold standard for assessing glycaemic |
| Hybrid (RRF) | 4 | False | idf-clinical-practice-recommendations-2025#213 | 5%–7.1% (48–54 mmol/mol).4-8 These data are the basis for the general target of HbA1c <7.0% (<53 mmol/mol) in people with T2DM recommended by most guidelines for minimising risk of diabetes complications and hypoglycaemia. This target is proposed as a guide, which is appropriate for many people, but it is important to consider personalising targets, |
| Hybrid (RRF) | 5 | False | sign154-pharmacological-glycaemic-control#163 | diabetes is reasonable to reduce the risk of microvascular and macrovascular disease. A target of 6.5% (48 mmol/mol) may be appropriate at diagnosis. Targets should be set with individuals in order to balance benefits with harms, in particular hypoglycaemia and weight gain. 3 • Targets for glycaemic control 10 / 4 Metformin Metformin is a small molecule from the biguanide family that has been used as a |

## Part C - Reranking (4 marks)

Rerank the best Part B method's top ten candidates with a cross-encoder and retain three. Compare rankings, hit rate@1, and scoring latency. Rank change rate is the share of queries whose first result changes.


### C1 Rerank (1 mark)

Brief: "Rerank: take your best method from Part B and retrieve the top 10 for each query. Re-score them with a cross-encoder, for example cross-encoder/ms-marco-MiniLM-L-6-v2, and keep the top 3. (1 mark)"

In [22]:
# C1: rerank the top 10 to the top 3.
import torch
from sentence_transformers import CrossEncoder

RERANK_MODEL_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"
with working("Loading the cross-encoder"):
    ce = CrossEncoder(RERANK_MODEL_ID, device=L.DEVICE)
print(f"cross-encoder: {RERANK_MODEL_ID} on {L.DEVICE}")

best_fn = {"Dense": dense_fn, "BM25": bm25_fn, "Hybrid (RRF)": hybrid_fn}[BEST_METHOD]

# Exclude one warm-up; keep both rankings for the per-query comparison.
ce.predict([(QUERIES[0]["query"], "warm-up")])
final_chunks, before_top3, rerank_ms = {}, {}, []
for item in QUERIES:
    before, after, seconds = rerank_top3(item["query"])
    before_top3[item["qid"]] = before
    final_chunks[item["qid"]] = after
    rerank_ms.append(seconds * 1000)
print(f"reranked {len(QUERIES)} queries; kept the top 3 of 10 for each")


Loading the cross-encoder...
Loading the cross-encoder: completed in 1.3s
cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2 on cuda
reranked 10 queries; kept the top 3 of 10 for each


### C2 Compare rankings (1 mark)


In [23]:
# C2: top 3 before and after reranking, and the rank change rate.
changed = 0
comparison_rows = []
for item in QUERIES:
    qid = item["qid"]
    for stage, ranking in (("Before", before_top3[qid]), ("After", final_chunks[qid])):
        for chunk in ranking:
            comparison_rows.append((f"{qid}: {item['query']}", stage, str(chunk["rank"]), chunk["chunk_id"],
                                    str(is_hit(chunk["text"], item["span"])), chunk["text"]))
    changed += before_top3[qid][0]["chunk_id"] != final_chunks[qid][0]["chunk_id"]
RANK_CHANGE = 100 * changed / len(QUERIES)
show_md_table(["Query", "Stage", "Rank", "Chunk", "Span hit", "Full chunk text"],
              comparison_rows, title="C2 top three before and after reranking")
print(f"Rank change rate: {RANK_CHANGE:.0f}% ({changed}/{len(QUERIES)} first results changed)")


**C2 top three before and after reranking**

| Query | Stage | Rank | Chunk | Span hit | Full chunk text |
| --- | --- | --- | --- | --- | --- |
| Q1: What should be considered when planning detection of undiagnosed T2DM and prevention programmes in low- and middle-income countries? | Before | 1 | idf-clinical-practice-recommendations-2025#53 | True | oglobin (HbA1c). Programmes for the detection of undiagnosed T2DM and the prevention of progression to T2DM in highrisk individuals are well-accepted strategies to reduce the burden of diabetes. GLOBAL CONSIDERATIONS The capacity to detect individuals with undiagnosed T2DM and IH and implement diabetes prevention programmes varies considerably between countries and health services and systems, especially in low- and middle-income countries (LMIC). The World Health Organization ( |
| Q1: What should be considered when planning detection of undiagnosed T2DM and prevention programmes in low- and middle-income countries? | Before | 2 | idf-clinical-practice-recommendations-2025#70 | False | 251.7 million) adults aged 20–79 years old living with diabetes were undiagnosed. There were significant regional disparities in undiagnosed diabetes, and 86.9% of all undiagnosed cases were in low- and middle-income countries. Low-income countries had the highest proportion of undiagnosed cases (58.7%), followed by middle-income countries at 45.5%, and high-income countries at 28.9% |
| Q1: What should be considered when planning detection of undiagnosed T2DM and prevention programmes in low- and middle-income countries? | Before | 3 | idf-clinical-practice-recommendations-2025#117 | False | less than in RCTs and data on incident diabetes are lacking. Including strategies to reduce population diabetes risk is an important component of diabetes prevention programmes. Implementing diabetes prevention programmes Diabetes prevention programmes should ideally include both high-risk and population strategies. Encouragingly, the incidence of diagnosed diabetes appears to be stabilising or declining in many higher-income countries, with a reported annual estimated change in incidence ranging from –1.1% to –10.8% since 2010. Multifaceted T2DM |
| Q1: What should be considered when planning detection of undiagnosed T2DM and prevention programmes in low- and middle-income countries? | After | 1 | idf-clinical-practice-recommendations-2025#53 | True | oglobin (HbA1c). Programmes for the detection of undiagnosed T2DM and the prevention of progression to T2DM in highrisk individuals are well-accepted strategies to reduce the burden of diabetes. GLOBAL CONSIDERATIONS The capacity to detect individuals with undiagnosed T2DM and IH and implement diabetes prevention programmes varies considerably between countries and health services and systems, especially in low- and middle-income countries (LMIC). The World Health Organization ( |
| Q1: What should be considered when planning detection of undiagnosed T2DM and prevention programmes in low- and middle-income countries? | After | 2 | idf-clinical-practice-recommendations-2025#117 | False | less than in RCTs and data on incident diabetes are lacking. Including strategies to reduce population diabetes risk is an important component of diabetes prevention programmes. Implementing diabetes prevention programmes Diabetes prevention programmes should ideally include both high-risk and population strategies. Encouragingly, the incidence of diagnosed diabetes appears to be stabilising or declining in many higher-income countries, with a reported annual estimated change in incidence ranging from –1.1% to –10.8% since 2010. Multifaceted T2DM |
| Q1: What should be considered when planning detection of undiagnosed T2DM and prevention programmes in low- and middle-income countries? | After | 3 | who-global-report-diabetes-2016#317 | False | or standards for diabetes management. Implementation and funding of national policies and guidelines is uneven. Basic technologies for early detection, diagnosis and monitoring of diabetes in primary care settings are generally not available in low-income and lower middleincome countries. Availability of insulin, metformin and sulphonylurea(s) is very limited in primary care facilities in low-income countries. Less than half of countries have conducted a national, population-based survey with measurement of blood glucose status within the past |
| Q2: Which foods should people with diabetes and dyslipidaemia restrict because of their saturated and trans fats? | Before | 1 | icmr-type2-diabetes-guidelines-2018#65 | True | . Use of fruits and vegetables should be encouraged, with good quality fats in moderation. (b) Dyslipidaemia : Saturated and trans fats food sources like vanaspati, butter, ghee, margarine, coconut oil, red meats like sausages, ham, bacon, egg yellow, whole milk and its products should be restricted. Use of healthy oils and fibre rich foods is recommended. Vegetarians can take flax seeds (10 g / day) in |
| Q2: Which foods should people with diabetes and dyslipidaemia restrict because of their saturated and trans fats? | Before | 2 | moh-malaysia-type2-diabetes-cpg-2020#177 | False | 81.0 g/kg body weight/day may be recommended.115 (Level I) MANAGEMENT OF TYPE 2 DIABETES MELLITUS v. Fats - Patients with T2DM should limit total fat (25%-35% energy intake), saturated fats (<7% energy intake) and minimise trans-fat (<1% energy intake) for prevention and treatment of CVD.75,116 (Level I) - A healthy diet incorporating oats, nuts and leg |
| Q2: Which foods should people with diabetes and dyslipidaemia restrict because of their saturated and trans fats? | Before | 3 | racgp-gp-management-type2-diabetes-2016#227 | False | A0alcohol: – Restrict foods high in saturated fat such as many biscuits, cakes, pastries, pies, processed meats, commercial burgers, pizza, fried foods, potato chips, crisps and other savoury snacks. – Limit foods and drinks containing added salt. – Avoid foods and drinks containing added sugars such as confectionary, sugarsweetened soft drinks and cordials, fruit drinks, vitamin waters, energy and sports drinks. – Alcohol intake |
| Q2: Which foods should people with diabetes and dyslipidaemia restrict because of their saturated and trans fats? | After | 1 | icmr-type2-diabetes-guidelines-2018#65 | True | . Use of fruits and vegetables should be encouraged, with good quality fats in moderation. (b) Dyslipidaemia : Saturated and trans fats food sources like vanaspati, butter, ghee, margarine, coconut oil, red meats like sausages, ham, bacon, egg yellow, whole milk and its products should be restricted. Use of healthy oils and fibre rich foods is recommended. Vegetarians can take flax seeds (10 g / day) in |
| Q2: Which foods should people with diabetes and dyslipidaemia restrict because of their saturated and trans fats? | After | 2 | racgp-gp-management-type2-diabetes-2016#227 | False | A0alcohol: – Restrict foods high in saturated fat such as many biscuits, cakes, pastries, pies, processed meats, commercial burgers, pizza, fried foods, potato chips, crisps and other savoury snacks. – Limit foods and drinks containing added salt. – Avoid foods and drinks containing added sugars such as confectionary, sugarsweetened soft drinks and cordials, fruit drinks, vitamin waters, energy and sports drinks. – Alcohol intake |
| Q2: Which foods should people with diabetes and dyslipidaemia restrict because of their saturated and trans fats? | After | 3 | moh-malaysia-type2-diabetes-cpg-2020#177 | False | 81.0 g/kg body weight/day may be recommended.115 (Level I) MANAGEMENT OF TYPE 2 DIABETES MELLITUS v. Fats - Patients with T2DM should limit total fat (25%-35% energy intake), saturated fats (<7% energy intake) and minimise trans-fat (<1% energy intake) for prevention and treatment of CVD.75,116 (Level I) - A healthy diet incorporating oats, nuts and leg |
| Q3: How much flax seed can vegetarians take daily to obtain omega-3 fatty acids? | Before | 1 | icmr-type2-diabetes-guidelines-2018#65 | False | . Use of fruits and vegetables should be encouraged, with good quality fats in moderation. (b) Dyslipidaemia : Saturated and trans fats food sources like vanaspati, butter, ghee, margarine, coconut oil, red meats like sausages, ham, bacon, egg yellow, whole milk and its products should be restricted. Use of healthy oils and fibre rich foods is recommended. Vegetarians can take flax seeds (10 g / day) in |
| Q3: How much flax seed can vegetarians take daily to obtain omega-3 fatty acids? | Before | 2 | icmr-type2-diabetes-guidelines-2018#66 | False | their diet as both fish and flax seeds are rich in omega-3 fatty acids which is protective for heart disease. Alcohol restriction will bring down the triglycerides. Dietary management should be accompanied with regular physical activity and exercise regimen. 6.2.3 Special situations requiring dietary modification: (i) Sick Days In the event of fever or other illness, the diabetic diet should be modified by changing the consistency and texture of foods to maintain adequate calorie intake. Semi solid |
| Q3: How much flax seed can vegetarians take daily to obtain omega-3 fatty acids? | Before | 3 | hse-ireland-nutrition-dietetic-type2-diabetes-2020#144 | False | educates people on dietary strategies to increase omega -3 intake from foods as part of a healthy balanced diet. FSAI 2019 state oily fish is the best food source of long chain omega 3 fatty acids (EPA & DHA). See oily fish section above for more information. Regarding supplements: Re CVD risk: Routine use of n -3 fish oil supplements is not recommended for CVD risk reduction in Type 2 Diabetes at this time (Evert et al 2019, ADA 2019, |
| Q3: How much flax seed can vegetarians take daily to obtain omega-3 fatty acids? | After | 1 | icmr-type2-diabetes-guidelines-2018#65 | False | . Use of fruits and vegetables should be encouraged, with good quality fats in moderation. (b) Dyslipidaemia : Saturated and trans fats food sources like vanaspati, butter, ghee, margarine, coconut oil, red meats like sausages, ham, bacon, egg yellow, whole milk and its products should be restricted. Use of healthy oils and fibre rich foods is recommended. Vegetarians can take flax seeds (10 g / day) in |
| Q3: How much flax seed can vegetarians take daily to obtain omega-3 fatty acids? | After | 2 | hse-ireland-nutrition-dietetic-type2-diabetes-2020#139 | False | (DUK 2018). Note this is higher than what is recommended for the general population.  Plant sources of n -3, alpha-linolenic acid (ALA), have been shown to be associated with lower risk of CVD (DUK 2018).  Vegetarian sources of PUFA n-3 are found in walnuts, flax seeds, chia seeds, rapeseed oil and soya bean oil, however dietary ALA has a low conversion rate to EPA/DHA ( |
| Q3: How much flax seed can vegetarians take daily to obtain omega-3 fatty acids? | After | 3 | icmr-type2-diabetes-guidelines-2018#66 | False | their diet as both fish and flax seeds are rich in omega-3 fatty acids which is protective for heart disease. Alcohol restriction will bring down the triglycerides. Dietary management should be accompanied with regular physical activity and exercise regimen. 6.2.3 Special situations requiring dietary modification: (i) Sick Days In the event of fever or other illness, the diabetic diet should be modified by changing the consistency and texture of foods to maintain adequate calorie intake. Semi solid |
| Q4: Why does WHO conduct a periodic survey of country capacity for noncommunicable diseases? | Before | 1 | idf-clinical-practice-recommendations-2025#54 | False | WHO) periodically conducts a country capacity survey on noncommunicable diseases (NCDs) to assess the national-level response to the NCD burden, which includes information on health system capacity for diabetes detection, treatment, and care. The most recent survey in 2019 collected data from 160 countries; 75% reported having an operational policy, strategy, or action plan for diabetes. Specific details on diabetes screening and prevention were not reported. The percent of countries with guidelines utilised in at least 50% |
| Q4: Why does WHO conduct a periodic survey of country capacity for noncommunicable diseases? | Before | 2 | who-global-report-diabetes-2016#319 | False | Declaration and the 2014 United Nations General Assembly Outcome Document on Noncommunicable Diseases (1). Data presented in this chapter come from the 2015 Noncommunicable Disease Country Capacity Survey (NCD CCS), to which national teams from 177 WHO Member States, representing 97% of the world’s population, responded. The NCD CCS has been conducted regularly since 2000 to assess NCD governance and infrastructure, policy response, surveillance and health systems response at country level. It |
| Q4: Why does WHO conduct a periodic survey of country capacity for noncommunicable diseases? | Before | 3 | who-global-report-diabetes-2016#396 | False | TO DIABETES P rEvENTION AND CONT rOl Assessment of national capacity indicators related to diabetes prevention management was based on Member State responses to the 2015 Noncommunicable Disease Country Capacity Survey (NCD CCS) (13). The NCD CCS is conducted periodically by WHO to assess individual country capacity for NCD prevention. A first survey was conducted in 2000, followed by surveys in 2005, 2010, 2013 and 2015. The questionnaire covers health system infrastructure; funding; policies |
| Q4: Why does WHO conduct a periodic survey of country capacity for noncommunicable diseases? | After | 1 | idf-clinical-practice-recommendations-2025#54 | False | WHO) periodically conducts a country capacity survey on noncommunicable diseases (NCDs) to assess the national-level response to the NCD burden, which includes information on health system capacity for diabetes detection, treatment, and care. The most recent survey in 2019 collected data from 160 countries; 75% reported having an operational policy, strategy, or action plan for diabetes. Specific details on diabetes screening and prevention were not reported. The percent of countries with guidelines utilised in at least 50% |
| Q4: Why does WHO conduct a periodic survey of country capacity for noncommunicable diseases? | After | 2 | who-global-report-diabetes-2016#396 | False | TO DIABETES P rEvENTION AND CONT rOl Assessment of national capacity indicators related to diabetes prevention management was based on Member State responses to the 2015 Noncommunicable Disease Country Capacity Survey (NCD CCS) (13). The NCD CCS is conducted periodically by WHO to assess individual country capacity for NCD prevention. A first survey was conducted in 2000, followed by surveys in 2005, 2010, 2013 and 2015. The questionnaire covers health system infrastructure; funding; policies |
| Q4: Why does WHO conduct a periodic survey of country capacity for noncommunicable diseases? | After | 3 | idf-clinical-practice-recommendations-2025#197 | False | is available throughout their services. However, at present it is not available in many health services. The 2019 periodic WHO country capacity survey on non-communicable diseases (NCDs) of 160 countries reported that overall HbA1c testing was generally available in only 53% of primary care facilities in the public and private health sectors.19 The Pan American Health Organization separately reported that 18 of 35 countries (51.4%) had HbA1c testing generally available in >50% primary |
| Q5: How should glycaemic targets be set for people with T2DM? | Before | 1 | sign154-pharmacological-glycaemic-control#138 | False | group as the key clinical recommendations that should be prioritised for implementation. 2.1 TARGETS FOR GLYCAEMIC CONTROL R An HbA1c target of 7.0% (53 mmol/mol) among people with type 2 diabetes is reasonable to reduce the risk of microvascular and macrovascular disease. A target of 6.5% (48 mmol/mol) may be appropriate at diagnosis. Targets should be set with individuals in order to balance benefits |
| Q5: How should glycaemic targets be set for people with T2DM? | Before | 2 | moh-malaysia-type2-diabetes-cpg-2020#392 | True | emic control, co-morbidities, complications and other CVD risk factors should be done at initial visit and whenever indicated subsequently. Grade C 1. Glycaemic targets must be individualised. Therapy in most patients with T2DM should be targeted to achieve an HbA1c≤7.0% or 6.5% (where appropriate), if achievable without significant hypoglycaemia. Reduction in HbA1c has been shown to |
| Q5: How should glycaemic targets be set for people with T2DM? | Before | 3 | idf-clinical-practice-recommendations-2025#181 | False | GLYCAEMIC TARGETS o Glycaemic control reduces the risk of both micro- and macrovascular complications in type 2 diabetes mellitus (T2DM). o Assessing and monitoring glycaemia is an essential component of guiding treatment decisions to achieve and maintain target glycaemic control. o Setting glycaemic targets is based on established associations with adverse outcomes. o HbA1c is the established gold standard for assessing glycaemic |
| Q5: How should glycaemic targets be set for people with T2DM? | After | 1 | moh-malaysia-type2-diabetes-cpg-2020#392 | True | emic control, co-morbidities, complications and other CVD risk factors should be done at initial visit and whenever indicated subsequently. Grade C 1. Glycaemic targets must be individualised. Therapy in most patients with T2DM should be targeted to achieve an HbA1c≤7.0% or 6.5% (where appropriate), if achievable without significant hypoglycaemia. Reduction in HbA1c has been shown to |
| Q5: How should glycaemic targets be set for people with T2DM? | After | 2 | sign154-pharmacological-glycaemic-control#138 | False | group as the key clinical recommendations that should be prioritised for implementation. 2.1 TARGETS FOR GLYCAEMIC CONTROL R An HbA1c target of 7.0% (53 mmol/mol) among people with type 2 diabetes is reasonable to reduce the risk of microvascular and macrovascular disease. A target of 6.5% (48 mmol/mol) may be appropriate at diagnosis. Targets should be set with individuals in order to balance benefits |
| Q5: How should glycaemic targets be set for people with T2DM? | After | 3 | idf-clinical-practice-recommendations-2025#213 | False | 5%–7.1% (48–54 mmol/mol).4-8 These data are the basis for the general target of HbA1c <7.0% (<53 mmol/mol) in people with T2DM recommended by most guidelines for minimising risk of diabetes complications and hypoglycaemia. This target is proposed as a guide, which is appropriate for many people, but it is important to consider personalising targets, |
| Q6: How many data sources underpinned the IDF diabetes prevalence estimates, and what period did the research cover? | Before | 1 | idf-atlas-global-diabetes-prevalence-2021#20 | False | was based on predefined criteria, listed in Table 1. The complete set of data sources for the estimation of diabetes prevalence included new studies retrieved from the 2020 literature search or the grey literature, as well as eligible data sources from the previous IDF Diabetes Atlas editions. To ensure that our modelling of diabetes prevalence estimates reflects the most recent data sources, studies conducted prior to 2005 were excluded except in circumstances were there was no other credible in-country data and extrapolation from nearby countries provided implaus |
| Q6: How many data sources underpinned the IDF diabetes prevalence estimates, and what period did the research cover? | Before | 2 | idf-atlas-global-diabetes-prevalence-2021#59 | False | but represents an area where additional research would be informative. The discordance between the large projected increase in diabetes cases and the only slight increase in projected diabetes-related health expenditures occurs because most of the increase in diabetes cases is in low- and middle-income countries in which expenditures related to diabetes are much lower. 4.1. Limitations The IDF estimates of diabetes prevalence are not without limitations. With each new IDF Diabetes Atlas edition, our methods are reviewed and improvements are made where possible. For |
| Q6: How many data sources underpinned the IDF diabetes prevalence estimates, and what period did the research cover? | Before | 3 | idf-atlas-global-diabetes-prevalence-2021#39 | False | met our inclusion criteria and were added into the IDF Diabetes Atlas database. The grey literature search (including data from the IDF Regional offices and WHO STEPwise approach to Surveillance (STEPS) studies led to the identification of an additional 53 new eligible data sources. The analytic hierarchy process was then applied to the data sources retrieved from literature search, grey literature, WHO STEPS studies and existing studies in the databases. From this process, a total of 219 data sources from 144 countries were selected for inclusion. Among these, |
| Q6: How many data sources underpinned the IDF diabetes prevalence estimates, and what period did the research cover? | After | 1 | idf-atlas-global-diabetes-prevalence-2021#20 | False | was based on predefined criteria, listed in Table 1. The complete set of data sources for the estimation of diabetes prevalence included new studies retrieved from the 2020 literature search or the grey literature, as well as eligible data sources from the previous IDF Diabetes Atlas editions. To ensure that our modelling of diabetes prevalence estimates reflects the most recent data sources, studies conducted prior to 2005 were excluded except in circumstances were there was no other credible in-country data and extrapolation from nearby countries provided implaus |
| Q6: How many data sources underpinned the IDF diabetes prevalence estimates, and what period did the research cover? | After | 2 | idf-atlas-global-diabetes-prevalence-2021#42 | False | total of 48 (21.9%) sources captured diabetes using self-report only. The list of sources used can be found at www.diabetesatlas.org. The number of original data sources from each IDF Region varied. The SEA Region had the most countries (85.7%, 18 out of 21 countries) with at least one in-country data source while the AFR Region the least (41.7%, 20 out of 48 countries). Data sources were available in |
| Q6: How many data sources underpinned the IDF diabetes prevalence estimates, and what period did the research cover? | After | 3 | idf-atlas-global-diabetes-prevalence-2021#66 | False | studies published before 2005. Since older studies generally tend to report lower prevalence, the exclusion of these studies may have contributed to the 12.9% rise in global prevalence when compared to the prevalence estimate from the 9th edition of Atlas. As with earlier IDF Diabetes Atlas editions, our aim was to produce what we believe is the best estimate of diabetes prevalence for all countries and territories with a population of at least 50,000 persons. Therefore, when a country lacked internal data on diabetes prevalence, this was estimated |
| Q7: How should people starting a sulphonylurea be counselled about hypoglycaemia warning signs? | Before | 1 | sign154-pharmacological-glycaemic-control#378 | True | , polyuria and weight loss. They are more effective early in the management of type 2 diabetes. Their main side effects are hypoglycaemia and weight gain. The warning signs of hypoglycaemia, which should be outlined to people taking these agents, include (early signs) tremor, sweating, shaking, irritability, and (later signs) lack of concentration and co-ordination. People prescribed sulphonylureas should be prescribed adequate supplies of blood glucose test |
| Q7: How should people starting a sulphonylurea be counselled about hypoglycaemia warning signs? | Before | 2 | racgp-gp-management-type2-diabetes-2016#505 | False | glucose states can occur. Both have signi/f_icant impacts and implications, and require patient knowledge and active management planning. Hypoglycaemia occurs in people with diabetes when their blood glucose level falls below 4 mmol/L or is at a level that causes symptoms and signs. It is more common in people taking insulin. However, it can also occur with sulphonylurea therapy either alone or supplementing other oral therapies. Other causative factors are |
| Q7: How should people starting a sulphonylurea be counselled about hypoglycaemia warning signs? | Before | 3 | racgp-gp-management-type2-diabetes-2016#509 | False | c thresholds for noti/f_ication and intervention may be considered in management planning (eg SMBG >15mmol/L on two subsequent occasions, two hours apart). In practice All patients with type 2 diabetes on insulin and/or sulphonylureas and their families need to be informed about the risks, signs and symptoms, and actions to be taken. If/uni00A0there has been severe hypoglycaemia, id |
| Q7: How should people starting a sulphonylurea be counselled about hypoglycaemia warning signs? | After | 1 | sign154-pharmacological-glycaemic-control#378 | True | , polyuria and weight loss. They are more effective early in the management of type 2 diabetes. Their main side effects are hypoglycaemia and weight gain. The warning signs of hypoglycaemia, which should be outlined to people taking these agents, include (early signs) tremor, sweating, shaking, irritability, and (later signs) lack of concentration and co-ordination. People prescribed sulphonylureas should be prescribed adequate supplies of blood glucose test |
| Q7: How should people starting a sulphonylurea be counselled about hypoglycaemia warning signs? | After | 2 | racgp-gp-management-type2-diabetes-2016#368 | False | be taken with those at increased risk of hypoglycaemia, especially the elderly. People taking sulphonylureas or insulin may need to notify motor vehicle licensing authorities as these medications can affect driving performance, as well as increase the patient’s burden by requiring glucose self monitoring, especially on initiation or dose titrations. It/uni00A0is important that patients inform their insurance agents or companies (also refer to Chapter 14. Management of other impacts of diabetes) |
| Q7: How should people starting a sulphonylurea be counselled about hypoglycaemia warning signs? | After | 3 | waikato-gp-type2-diabetes-manual-2024#321 | False | weeks or months. This helps ‘reset’ their body warning system so that they start experiencing symptoms of hypos when their blood glucose levels fall below 4mmols/L. Alcohol Alcohol is very energy dense, and it is easy to gain weight if drinking regularly. • Alcohol inhibits hepatic gluconeogenesis and induces hypoglycaemia whenever gluconeogenesis is required to maintain normal glucose levels. People with diabetes on a sulphonylurea or |
| Q8: How should the diet be changed during fever or other illness in diabetes? | Before | 1 | icmr-type2-diabetes-guidelines-2018#66 | True | their diet as both fish and flax seeds are rich in omega-3 fatty acids which is protective for heart disease. Alcohol restriction will bring down the triglycerides. Dietary management should be accompanied with regular physical activity and exercise regimen. 6.2.3 Special situations requiring dietary modification: (i) Sick Days In the event of fever or other illness, the diabetic diet should be modified by changing the consistency and texture of foods to maintain adequate calorie intake. Semi solid |
| Q8: How should the diet be changed during fever or other illness in diabetes? | Before | 2 | waikato-gp-type2-diabetes-manual-2024#363 | False | illness can cause changes to blood glucose levels. Levels can go high or low depending on the type of illness. Infections that cause fever and pain often cause high blood glucose levels. Gastroenteritis (vomiting and diarrhoea) can cause low blood glucose levels. Vomiting can also be a sign of not enough insulin. If a child with type 1 diabetes presents with an acute illness: • Ask the care giver for the sick day management plan from their diabetes CNS • Never withhold insulin |
| Q8: How should the diet be changed during fever or other illness in diabetes? | Before | 3 | hse-ireland-nutrition-dietetic-type2-diabetes-2020#422 | False | a person is being treated for type 2 diabetes with insulin, education on ketone testing during illness may nee d to be considered depending on the individual needs of the client. General Guidelines during Illness  Diabetes medicine taken as prescribed.  Blood glucose levels should be checked more frequently if testing at home .  Eat normally when possible and keep hydrated by sipping on water and sugar free fluids .  If you are on insulin or tablets that have the potential to cause hypos and you are unable to eat it is important to |
| Q8: How should the diet be changed during fever or other illness in diabetes? | After | 1 | icmr-type2-diabetes-guidelines-2018#66 | True | their diet as both fish and flax seeds are rich in omega-3 fatty acids which is protective for heart disease. Alcohol restriction will bring down the triglycerides. Dietary management should be accompanied with regular physical activity and exercise regimen. 6.2.3 Special situations requiring dietary modification: (i) Sick Days In the event of fever or other illness, the diabetic diet should be modified by changing the consistency and texture of foods to maintain adequate calorie intake. Semi solid |
| Q8: How should the diet be changed during fever or other illness in diabetes? | After | 2 | racgp-gp-management-type2-diabetes-2016#235 | False | and/or hypoglycaemia.77 In practice Dietary habit changes are often slow and incremental. There is no need for a ‘special’ diet for diabetes, just the requirement to follow a sensible, balanced eating plan. Keep advice simple and educate patients about healthy food choices rather than on unhealthy food choices or what they should not eat. All sugars do not need to be eliminated. A small amount of sugar as part of a mixed meal or food (e |
| Q8: How should the diet be changed during fever or other illness in diabetes? | After | 3 | waikato-gp-type2-diabetes-manual-2024#335 | False | for people with type two diabetes. When you are sick you may notice that your blood glucose levels change. This can be because your appetite changes, but also because infections, trauma, stress and other illnesses can cause your blood glucose levels to go up or down. It can be difficult to manage your diabetes on your own when you feel crook, so it is important to let a whānau member, friend or neighbour know when you are unwell. Drink: If you are vomiting and not eating or |
| Q9: For how many days is each type of continuous glucose monitoring used when advanced monitoring is needed? | Before | 1 | icmr-type2-diabetes-guidelines-2018#48 | True | hypoglycaemia, severe fluctuations in blood glucose, disparity in measured glucose values and HbA1c, severe anaemia, haemoglobinopathy, hypoglycaemia unawareness etc. Here Continuous Glucose Monitoring (CGM) for 5-7 days or iSCGM (intermittently scanned CGM) for 14 days may be used. SECTION 6 NON-PHARMACOLOGICAL MANAGEMENT OF DIABETES 6.1. Lifestyle Goals in Diabetes z To improve |
| Q9: For how many days is each type of continuous glucose monitoring used when advanced monitoring is needed? | Before | 2 | idf-clinical-practice-recommendations-2025#317 | False | Visseren FLJ, et al. Continuous glucose monitoring in adults with type 2 diabetes: A systematic review and metaanalysis. Diabetologia. 2024;67(5):798-810. 77. Seidu S, Kunutsor SK, Ajjan RA, et al. Efficacy and safety of continuous glucose monitoring and intermittently scanned continuous glucose monitoring in patients with type 2 diabetes: A systematic review and metaanalysis of interventional evidence. Diabetes Care. 202 |
| Q9: For how many days is each type of continuous glucose monitoring used when advanced monitoring is needed? | Before | 3 | moh-malaysia-type2-diabetes-cpg-2020#1174 | False | H, et al. CONTINUOUS GLUCOSE MONITORING: A CONSENSUS CONFERENCE OF THE AMERICAN ASSOCIATION OF CLINICAL ENDOCRINOLOGISTS AND AMERICAN COLLEGE OF ENDOCRINOLOGY. Endocr Pract. 2016;22(8):1008-1021. 350. Danne T, Nimri R, Battelino T, et al. International Consensus on Use of Continuous Glucose Monitoring. Diabetes Care. 2017;40(12):1631-1640. 351. Petrie JR, Peters |
| Q9: For how many days is each type of continuous glucose monitoring used when advanced monitoring is needed? | After | 1 | icmr-type2-diabetes-guidelines-2018#48 | True | hypoglycaemia, severe fluctuations in blood glucose, disparity in measured glucose values and HbA1c, severe anaemia, haemoglobinopathy, hypoglycaemia unawareness etc. Here Continuous Glucose Monitoring (CGM) for 5-7 days or iSCGM (intermittently scanned CGM) for 14 days may be used. SECTION 6 NON-PHARMACOLOGICAL MANAGEMENT OF DIABETES 6.1. Lifestyle Goals in Diabetes z To improve |
| Q9: For how many days is each type of continuous glucose monitoring used when advanced monitoring is needed? | After | 2 | idf-clinical-practice-recommendations-2025#246 | False | An international consensus has proposed targets for key glycaemic metrics as summarised in Table 2.2.58 Table 2.2 Key metrics for assessing glycaemic status using continuous glucose monitoring (CGM) in non-pregnant adults with diabetes58 Metric Interpretation Goal Number of days of CGM 14 days Percentage of time CGM device is active 70% data Coefficient of variation Percentage coefficient of variation Intraday (i.e., within 24 h) and Inter |
| Q9: For how many days is each type of continuous glucose monitoring used when advanced monitoring is needed? | After | 3 | nice-ng28-type2-diabetes-management#426 | False | cerebrovascular disease such as transient ischaemic attack and ischaemic stroke • peripheral arterial disease. Continuous glucose monitoring This covers both real-time continuous glucose monitoring (rtCGM) and intermittently scanned continuous glucose monitoring (isCGM, commonly referred to as 'flash'). A continuous glucose monitor is a device that measures blood glucose levels and sends the readings to a display device or smartphone. Early onset type 2 diabetes Diabetes that has been diagnosed before the age of |
| Q10: Which foods and fluids should be encouraged for people with diabetes on sick days? | Before | 1 | icmr-type2-diabetes-guidelines-2018#66 | False | their diet as both fish and flax seeds are rich in omega-3 fatty acids which is protective for heart disease. Alcohol restriction will bring down the triglycerides. Dietary management should be accompanied with regular physical activity and exercise regimen. 6.2.3 Special situations requiring dietary modification: (i) Sick Days In the event of fever or other illness, the diabetic diet should be modified by changing the consistency and texture of foods to maintain adequate calorie intake. Semi solid |
| Q10: Which foods and fluids should be encouraged for people with diabetes on sick days? | Before | 2 | hse-ireland-nutrition-dietetic-type2-diabetes-2020#416 | False | motor insurance c over. For further information see www.diabetes.ie or call Diabetes Ireland on 01-8428118. 12. ILLNESS – SICK DAY GUIDANCE Evidence The Model of Integrated care (2018) has included illness/sick days as one of the topics that should be covered by the practice nurse during the initial and review consultation with person with Type 2 diabetes. It is also included as one of the topics to be c overed by the Dietitian over a series of ses si |
| Q10: Which foods and fluids should be encouraged for people with diabetes on sick days? | Before | 3 | hse-ireland-nutrition-dietetic-type2-diabetes-2020#417 | False | ons, depending on the per son’s needs. The ICGP (2016) recommend that sick days are a key selfcare issue to be covered with a person as part of structured education programmes. In terms of guidance specifics, current available references are from the ADA 2019 and guidance from Diabetes & Primary Care in 2018. There is no guidance in Diabetes UK 2018 regarding illness management/ sick day guidance and there are no recommendations for DKA. Guidance The following summarises current guidance.  Any condition |
| Q10: Which foods and fluids should be encouraged for people with diabetes on sick days? | After | 1 | hse-ireland-nutrition-dietetic-type2-diabetes-2020#500 | False | with diabetes should be encouraged and assisted to achieve and maintain a healthy body weight. A consistent amount of carbohydrate should be provided at each meal. The meal plan can include sugar in moderate amounts but excess sugar, soft drinks and fruit juices should be avoided. Functionally dependent Encourage the consumption of adequate amounts of fluid to avoid dehydration especially in hot weather. Education and training are essential to enable healthcare professionals and caregivers to provide nutritional support. Sub-category A: |
| Q10: Which foods and fluids should be encouraged for people with diabetes on sick days? | After | 2 | icmr-type2-diabetes-guidelines-2018#66 | False | their diet as both fish and flax seeds are rich in omega-3 fatty acids which is protective for heart disease. Alcohol restriction will bring down the triglycerides. Dietary management should be accompanied with regular physical activity and exercise regimen. 6.2.3 Special situations requiring dietary modification: (i) Sick Days In the event of fever or other illness, the diabetic diet should be modified by changing the consistency and texture of foods to maintain adequate calorie intake. Semi solid |
| Q10: Which foods and fluids should be encouraged for people with diabetes on sick days? | After | 3 | waikato-gp-type2-diabetes-manual-2024#365 | False | sugar-free fluids • If the child has been vomiting or has diarrhoea, Pedialyte™ should be recommended by the and can be purchased from the pharmacy • Relieve the symptoms of pain and fever with Paracetamol or Ibuprofen Remember that teenagers who usually manage their diabetes independently need to be looked after when they are sick . THE YOUTH AND YOUNG ADULT DIABETES TEAM SEES YOUNG PEOPLE WITH TYPE 1 AND TYPE 2 DIABETES BETWEEN THE AGES OF 15 -25 FOR WRAP AROUND CARE. |

Rank change rate: 20% (2/10 first results changed)


### C3 Measure (1 mark)


In [24]:
# C3: reranking measures.
hit1_before = b_results[BEST_METHOD]["hit1"]
hit1_after = 100 * sum(
    is_hit(final_chunks[item["qid"]][0]["text"], item["span"])
    for item in QUERIES
) / len(QUERIES)
RERANK_MS = sum(rerank_ms) / len(rerank_ms)
print_table(["Measure", "Value"], [
    ("Reranking time per query (ms)", f"{RERANK_MS:.1f}"),
    ("Rank change rate (%)", f"{RANK_CHANGE:.0f}"),
    ("Hit rate@1 before reranking (from Part B, same method)", f"{hit1_before:.0f}"),
    ("Hit rate@1 after reranking", f"{hit1_after:.0f}"),
], title="Part C reranking measures")

Part C reranking measures
Measure                                                  Value
--------------------------------------------------------------
Reranking time per query (ms)                              8.0
Rank change rate (%)                                        20
Hit rate@1 before reranking (from Part B, same method)      50
Hit rate@1 after reranking                                  60


### C4 Did reranking help? (1 mark)


In [25]:
# C4: did reranking help?
improved, worsened = [], []
for item in QUERIES:
    qid = item["qid"]
    before_hit = is_hit(before_top3[qid][0]["text"], item["span"])
    after_hit = is_hit(final_chunks[qid][0]["text"], item["span"])
    if after_hit and not before_hit:
        improved.append(qid)
    elif before_hit and not after_hit:
        worsened.append(qid)
verdict = ("improved top-1 span retrieval" if hit1_after > hit1_before else
           "left aggregate top-1 span retrieval unchanged" if hit1_after == hit1_before else
           "reduced top-1 span retrieval accuracy")
display(Markdown(
    f"**C4:** Reranking {verdict}: hit rate@1 changed from {hit1_before:.0f}% to {hit1_after:.0f}% "
    f"at an additional {RERANK_MS:.1f} ms/query for cross-encoder scoring. "
    f"It gained top-1 hits for {', '.join(improved) or 'none'} and lost top-1 hits for "
    f"{', '.join(worsened) or 'none'}, as the full C2 rankings show. "
    f"Change in hit rate@1: {hit1_after-hit1_before:+.0f} percentage points, on these ten queries only."))


**C4:** Reranking improved top-1 span retrieval: hit rate@1 changed from 50% to 60% at an additional 8.0 ms/query for cross-encoder scoring. It gained top-1 hits for Q5 and lost top-1 hits for none, as the full C2 rankings show. Change in hit rate@1: +10 percentage points, on these ten queries only.

## Part D - Generate and Evaluate Answers (7 marks)

### D1 Grounded Generation (3 marks)

First run the Assignment 1 model with the three numbered, reranked chunks: show the prompt, all ten answers, and one out-of-corpus query. If it ignores the chunks or never says "Not found", switch to an instruct model as the PDF allows, using the same prompt and the same chunks.

In [26]:
# D setup: load the Assignment 1 generator (CPT checkpoint and QLoRA adapter, NF4).
import torch
import transformers
from packaging.version import Version
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not L.IS_GPU:
    raise RuntimeError("NF4 loading needs a CUDA GPU.")

# Transformers renamed this keyword in 4.56; support both API versions.
DTYPE_KW = "dtype" if Version(transformers.__version__) >= Version("4.56") else "torch_dtype"
COMPUTE_DTYPE = getattr(torch, L.DTYPE) if L.DTYPE in ("bfloat16", "float16") else torch.float16
# Weights are stored in 4-bit NF4; matrix maths runs in COMPUTE_DTYPE.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

A1_NAME = "mistralai/Mistral-7B-v0.1 + CPT + QLoRA adapter (Assignment 1)"
# The Assignment 1 model was trained on "Instruction: ... Response:" text, so it may run on into a new instruction.
A1_STOPS = ["Instruction:"]
with working("Loading the Assignment 1 generator"):
    # Load the continued-pretraining checkpoint first, then attach the trained adapter.
    a1_base = AutoModelForCausalLM.from_pretrained(
        CPT_DIR, quantization_config=bnb_config, **{DTYPE_KW: COMPUTE_DTYPE}, device_map={"": 0},
    )
    a1_model = PeftModel.from_pretrained(a1_base, ADAPTER_DIR).eval()
a1_tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)
print(f"Generator: {A1_NAME}; NF4 compute dtype: {COMPUTE_DTYPE}")


Loading the Assignment 1 generator...
Loading the Assignment 1 generator: completed in 216.9s
Generator: mistralai/Mistral-7B-v0.1 + CPT + QLoRA adapter (Assignment 1); NF4 compute dtype: torch.bfloat16


#### D1.1 Generation prompt (1 mark)

The prompt is defined in the shared setup cell in Part A, above A2.2. The next cell prints the template and one filled example.

In [27]:
# D1.1: print the grounding prompt template, then the filled prompt for the first query.
example = QUERIES[0]
example_chunks = [chunk["text"] for chunk in final_chunks[example["qid"]]]
print("Prompt template (GEN_PROMPT):")
print(GEN_PROMPT)
print(f"\nFilled prompt for {example['qid']}: {example['query']}")
print(build_generator_prompt(a1_tok, example["query"], example_chunks))


Prompt template (GEN_PROMPT):
Answer the question using ONLY the numbered chunks below.
Answer in full sentences. Cite the number of each chunk you used, for example [2], using only the numbers of the chunks below.
If the chunks do not contain the answer, reply exactly: Not found in the documents

Chunks:
[1] {chunk_1}
[2] {chunk_2}
[3] {chunk_3}

Question: {question}

Filled prompt for Q1: What should be considered when planning detection of undiagnosed T2DM and prevention programmes in low- and middle-income countries?
Instruction: Answer the question using ONLY the numbered chunks below.
Answer in full sentences. Cite the number of each chunk you used, for example [2], using only the numbers of the chunks below.
If the chunks do not contain the answer, reply exactly: Not found in the documents

Chunks:
[1] oglobin (HbA1c). Programmes
for the detection of undiagnosed T2DM and
the prevention of progression to T2DM in highrisk individuals are well-accepted strategies to
reduce the burd

#### D1.2 All 10 answers: Assignment 1 model (1 mark)

In [28]:
# D1.2 helper: grounded answer generation with greedy decoding and the answer cleanup.
NOT_FOUND = "Not found in the documents"
CITE = re.compile(r"\[(\d+)\]")

print(f"Not-found reply used for checks: {NOT_FOUND!r}")


Not-found reply used for checks: 'Not found in the documents'


In [29]:
# D setup: prompt budgets for the reranked chunks.
actual_budgets = []
for item in QUERIES:
    prompt = build_generator_prompt(a1_tok, item["query"], [chunk["text"] for chunk in final_chunks[item["qid"]]])
    length = len(a1_tok(prompt, add_special_tokens=False).input_ids)
    if length + GEN_MAX_NEW_TOKENS > GEN_CONTEXT_LIMIT:
        raise ValueError(f"{item['qid']}: reranked prompt exceeds the context limit.")
    actual_budgets.append((item["qid"], str(length), str(GEN_MAX_NEW_TOKENS)))
print_table(["Query", "Complete reranked prompt tokens", "Answer token budget"], actual_budgets,
            title="Part D actual prompt budgets (Assignment 1 tokenizer)")

Part D actual prompt budgets (Assignment 1 tokenizer)
Query   Complete reranked prompt tokens   Answer token budget
-------------------------------------------------------------
Q1                                  500                   256
Q2                                  487                   256
Q3                                  474                   256
Q4                                  506                   256
Q5                                  450                   256
Q6                                  491                   256
Q7                                  462                   256
Q8                                  454                   256
Q9                                  508                   256
Q10                                 493                   256


In [30]:
# D1.2: Assignment 1 model answers to all ten queries.
a1_answers = {}
for item in QUERIES:
    qid = item["qid"]
    chunk_texts = [chunk["text"] for chunk in final_chunks[qid]]
    a1_answers[qid] = generate_answer(a1_model, a1_tok, item["query"], chunk_texts, A1_STOPS)
    print(f"\n{qid}: {item['query']}\n  Answer: {a1_answers[qid]}\n  Chunks cited: {cited_numbers(a1_answers[qid])}")


Q1: What should be considered when planning detection of undiagnosed T2DM and prevention programmes in low- and middle-income countries?
  Answer: 1. The capacity to detect individuals with undiagnosed T2DM and IH and implement diabetes prevention programmes varies considerably between countries and health services and systems, especially in low- and middle-income countries. 2. The World Health Organization (WHO) less than in RCTs and data on incident diabetes are lacking. 3. Including strategies to reduce population diabetes risk is an important component of diabetes prevention programmes.
  Chunks cited: -

Q2: Which foods should people with diabetes and dyslipidaemia restrict because of their saturated and trans fats?
  Answer: 81.0 g/kg body weight/day of alcohol may be recommended.115 (Level I) The other restrictions are on foods high in saturated fat, added salt and added sugars.
  Chunks cited: -

Q3: How much flax seed can vegetarians take daily to obtain omega-3 fatty acids?


#### D1.3 A query whose answer is not in the corpus: Assignment 1 model (1 mark)

In [31]:
# D1.3: Assignment 1 model answers the out-of-corpus query.
OOC_QUERY = "Who won the 2018 FIFA World Cup?"
_, ooc_top3, _ = rerank_top3(OOC_QUERY)
ooc_chunk_texts = [chunk["text"] for chunk in ooc_top3]
for number, chunk in enumerate(ooc_top3, 1):
    preview = chunk['text'][:120] + ('...' if len(chunk['text']) > 120 else '')
    print(f"[{number}] {chunk['chunk_id']}: {preview}")
a1_ooc_answer = generate_answer(a1_model, a1_tok, OOC_QUERY, ooc_chunk_texts, A1_STOPS)
print(f"\nQuestion: {OOC_QUERY}\nReply: {a1_ooc_answer}")
print(f"Replied Not found: {'yes' if is_not_found(a1_ooc_answer) else 'no'}")

[1] idf-clinical-practice-recommendations-2025#6: Schwarz (Germany)
Luc Van Gaal (Belgium)
Antonio Pérez (Spain)
Gojka Roglic (Switzerland)
Banshi Saboo (India)
Petar M S...
[2] who-global-report-diabetes-2016#13: Gretchen Stevens, Meindert Van Hilten, Temo Waqanivalu and Stephen Whiting provided
technical input. The country profile...
[3] idf-atlas-global-diabetes-prevalence-2021#106: 3 Pakistan 33.0 3 Pakistan 62.2
4 USA 32.2 4 USA 36.3
5 Indonesia 19.5 5 Indonesia 28.6
6 Brazil 15.7 6 Brazil 23.2
7 Me...

Question: Who won the 2018 FIFA World Cup?
Reply: 2018 FIFA World Cup winner France, which beat Croatia 4-2 in the final.
Replied Not found: no


#### Switching the generator to Qwen2.5 7B Instruct

In the Assignment 1 run the model cited no chunks, and on the out-of-corpus query it answered from memory instead of replying "Not found in the documents". The assignment allows an instruct model in that case, and Qwen2.5 7B Instruct is one of the two models it lists. Qwen needs no gated access. It gets the same prompt and the same top 3 chunks, so its answers compare directly with the Assignment 1 answers. The judge is Llama 3.1 8B Instruct, a different model from the generator.


In [32]:
# D1 switch: show the problem seen with the Assignment 1 model.
a1_uncited = [qid for qid, answer in a1_answers.items() if not is_not_found(answer) and not has_valid_citation(answer)]
print("Problem seen with the Assignment 1 model:")
print(f"- {len(a1_uncited)} of 10 answers cite no chunk number ({', '.join(a1_uncited) or 'none'}).")
print(f"- Out-of-corpus query: {OOC_QUERY}\n  Reply: {a1_ooc_answer}\n  Replied Not found: "
      f"{'yes' if is_not_found(a1_ooc_answer) else 'no'}")
example_qid = a1_uncited[0] if a1_uncited else QUERIES[0]["qid"]
print(f"- Example answer ({example_qid}): {a1_answers[example_qid]}")

Problem seen with the Assignment 1 model:
- 10 of 10 answers cite no chunk number (Q1, Q2, Q3, Q4, Q5, Q6, Q7, Q8, Q9, Q10).
- Out-of-corpus query: Who won the 2018 FIFA World Cup?
  Reply: 2018 FIFA World Cup winner France, which beat Croatia 4-2 in the final.
  Replied Not found: no
- Example answer (Q1): 1. The capacity to detect individuals with undiagnosed T2DM and IH and implement diabetes prevention programmes varies considerably between countries and health services and systems, especially in low- and middle-income countries. 2. The World Health Organization (WHO) less than in RCTs and data on incident diabetes are lacking. 3. Including strategies to reduce population diabetes risk is an important component of diabetes prevention programmes.


In [33]:
# D1 switch: free memory and load Qwen2.5-7B-Instruct.
import gc

# Free the Assignment 1 model before loading the next 7B model.
a1_model = a1_base = None
gc.collect()
torch.cuda.empty_cache()

GENERATOR_ID = "Qwen/Qwen2.5-7B-Instruct"
with working("Loading the instruct generator"):
    gen_model, gen_tok = load_nf4(GENERATOR_ID)
print(f"Generator for the D1 answers: {GENERATOR_ID}")

Loading the instruct generator...
Loading the instruct generator: completed in 206.0s
Generator for the D1 answers: Qwen/Qwen2.5-7B-Instruct


#### D1.2 and D1.3 with Qwen2.5 7B Instruct

Same prompt and top 3 reranked chunks as the Assignment 1 run. These answers are judged in D2.


In [34]:
# D1.2: generate the answers to the ten queries with the instruct generator.
answers = {}
for item in QUERIES:
    qid = item["qid"]
    chunk_texts = [chunk["text"] for chunk in final_chunks[qid]]
    answers[qid] = generate_answer(gen_model, gen_tok, item["query"], chunk_texts)
    print(f"\n{qid}: {item['query']}\n  Answer: {answers[qid]}\n  Chunks cited: {cited_numbers(answers[qid])}")


Q1: What should be considered when planning detection of undiagnosed T2DM and prevention programmes in low- and middle-income countries?
  Answer: When planning detection of undiagnosed T2DM and prevention programmes in low- and middle-income countries, it is important to consider the varying capacities to detect individuals with undiagnosed T2DM and impaired glucose tolerance (IGT) and implement diabetes prevention programmes [1]. Additionally, there is a need to address the uneven implementation and funding of national policies and guidelines [3], as well as the limited availability of basic technologies for early detection, diagnosis, and monitoring of diabetes in primary care settings [1]. Furthermore, the limited availability of essential medications like insulin, metformin, and sulphonylureas in primary care facilities in low-income countries should also be taken into account [1]. Lastly, the fact that less than half of countries have conducted a national, population-based surve

In [35]:
# D1.3: the out-of-corpus query with the instruct generator, using the same chunks as the Assignment 1 run.
ooc_answer = generate_answer(gen_model, gen_tok, OOC_QUERY, ooc_chunk_texts)
print(f"\nOut-of-corpus question: {OOC_QUERY}\nReply: {ooc_answer}")
print(f"Replied Not found: {'yes' if is_not_found(ooc_answer) else 'no'}")


Out-of-corpus question: Who won the 2018 FIFA World Cup?
Reply: Not found in the documents
Replied Not found: yes


In [36]:
# D1 format audit: Not-found replies and citation-number format for each answer.
show_md_table(["Query", "Not found", "Citation format valid"],
              [(item["qid"], str(is_not_found(answers[item["qid"]])),
                "not applicable" if is_not_found(answers[item["qid"]]) else
                str(has_valid_citation(answers[item["qid"]]))) for item in QUERIES],
              title="D1 format audit (citation numbers alone do not prove support)")

**D1 format audit (citation numbers alone do not prove support)**

| Query | Not found | Citation format valid |
| --- | --- | --- |
| Q1 | False | True |
| Q2 | False | True |
| Q3 | False | True |
| Q4 | False | True |
| Q5 | False | True |
| Q6 | False | True |
| Q7 | False | True |
| Q8 | False | True |
| Q9 | False | True |
| Q10 | False | True |

### D2 Check the answers with an LLM judge (4 marks)

Judge: `meta-llama/Llama-3.1-8B-Instruct`, a different model from the Qwen generator. Llama 3.1 is gated, so `HF_TOKEN` must be set to a Hugging Face token with access to it. The judge cell stops with a message if no token is found.


In [37]:
# D2 setup: load the judge, a different model from the generator.
from huggingface_hub import get_token

# Move retrieval models off the GPU and release the generator.
embed_model.to("cpu")
ce.model.to("cpu")
gen_model = None
gc.collect()
torch.cuda.empty_cache()

# The judge must be a different model from the generator.
JUDGE_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
assert JUDGE_MODEL_ID != GENERATOR_ID, "The judge must differ from the generator."
# Llama 3.1 is gated; a Hugging Face token with access to it is required.
if get_token() is None:
    raise RuntimeError("HF_TOKEN is not set. A token with access to meta-llama/Llama-3.1-8B-Instruct is required.")
JUDGE_MAX_NEW_TOKENS = 256
with working("Loading the judge"):
    judge_model, judge_tok = load_nf4(JUDGE_MODEL_ID)
JUDGE_CONTEXT_LIMIT = int(judge_model.config.max_position_embeddings)
if 0 < judge_tok.model_max_length < 1_000_000:
    JUDGE_CONTEXT_LIMIT = min(JUDGE_CONTEXT_LIMIT, int(judge_tok.model_max_length))

print(f"Judge: {JUDGE_MODEL_ID}; NF4 storage; compute dtype: {COMPUTE_DTYPE}")
print(f"Judge context limit: {JUDGE_CONTEXT_LIMIT}; CUDA allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")


Loading the judge...
Loading the judge: completed in 61.1s
Judge: meta-llama/Llama-3.1-8B-Instruct; NF4 storage; compute dtype: torch.bfloat16
Judge context limit: 131072; CUDA allocated: 5.32 GiB


#### D2.1 Judge prompt and results (1 mark)


In [38]:
# D2.1: judge prompt.
JUDGE_PROMPT = """You are grading one answer about Type 2 Diabetes.

Question: {question}
Expected answer (reference): {expected}
Retrieved chunks:
[1] {chunk_1}
[2] {chunk_2}
[3] {chunk_3}
Model answer: {answer}

Check 1, CORRECT: compare the model answer with the expected answer.
  CORRECT = it states the same key fact as the expected answer.
  PARTLY CORRECT = it states only part of the key fact, or the right fact with a detail missing or wrong.
  WRONG = it states a different fact, contradicts the expected answer, says "Not found in the documents",
  or gives only numbers or citation markers with no answer sentence.
Check 2, SUPPORTED: can every statement in the model answer be found in the 3 chunks above?
  YES = every statement is in the chunks. NO = at least one statement is not.
  A "Not found in the documents" reply makes no claims, so it is SUPPORTED: YES.
  A reply of exactly "Not found in the documents" always gets CORRECT: WRONG and SUPPORTED: YES.
REASON: one sentence. If SUPPORTED is NO, the reason must quote the unsupported statement in double quotes.

Reply in exactly this format, with nothing else:
CORRECT: <CORRECT | PARTLY CORRECT | WRONG>
SUPPORTED: <YES | NO>
REASON: <one sentence>"""
print(JUDGE_PROMPT)


You are grading one answer about Type 2 Diabetes.

Question: {question}
Expected answer (reference): {expected}
Retrieved chunks:
[1] {chunk_1}
[2] {chunk_2}
[3] {chunk_3}
Model answer: {answer}

Check 1, CORRECT: compare the model answer with the expected answer.
  CORRECT = it states the same key fact as the expected answer.
  PARTLY CORRECT = it states only part of the key fact, or the right fact with a detail missing or wrong.
  WRONG = it states a different fact, contradicts the expected answer, says "Not found in the documents",
  or gives only numbers or citation markers with no answer sentence.
Check 2, SUPPORTED: can every statement in the model answer be found in the 3 chunks above?
  YES = every statement is in the chunks. NO = at least one statement is not.
  A "Not found in the documents" reply makes no claims, so it is SUPPORTED: YES.
  A reply of exactly "Not found in the documents" always gets CORRECT: WRONG and SUPPORTED: YES.
REASON: one sentence. If SUPPORTED is NO, 

In [39]:
# D2.1: judge helpers with retries and an audit file.

JUDGE_ATTEMPTS = []

print("Judge helpers ready")


Judge helpers ready


In [40]:
# D2.1: run the judge on all ten answers.
verdicts = {}
for item in QUERIES:
    qid = item["qid"]
    chunk_texts = [chunk["text"] for chunk in final_chunks[qid]]
    prompt = JUDGE_PROMPT.format(
        question=item["query"],
        expected=item["expected_answer"],
        chunk_1=chunk_texts[0],
        chunk_2=chunk_texts[1],
        chunk_3=chunk_texts[2],
        answer=answers[qid],
    )
    raw = judge_generate(prompt, has_claim(answers[qid]))
    verdicts[qid] = dict(parse_verdict(raw), raw=raw)
    verdict = verdicts[qid]
    print(f"{qid}: {verdict['correct']} / {verdict['supported']}")

Q1: CORRECT / YES
Q2: PARTLY CORRECT / YES
Q3: CORRECT / YES
Q4: CORRECT / YES
Q5: CORRECT / YES
Q6: PARTLY CORRECT / YES
Q7: PARTLY CORRECT / NO
Q8: CORRECT / YES
Q9: PARTLY CORRECT / NO
Q10: CORRECT / YES


In [41]:
# D2.1: check coverage and save the verdicts and answers.
expected_ids = {item["qid"] for item in QUERIES}
assert len(expected_ids) == 10 and set(verdicts) == expected_ids, "All ten queries must have judge verdicts."
assert set(answers) == expected_ids, "All ten generated answers are required."
assert all(valid_verdict(value) for value in verdicts.values()), "Resolve invalid judge responses before counting."
show_md_table(["Query", "Correct", "Supported", "Reason"],
              [(f"{item['qid']}: {item['query']}", verdicts[item["qid"]]["correct"],
                verdicts[item["qid"]]["supported"], verdicts[item["qid"]]["reason"]) for item in QUERIES],
              title="D2.1 judge results")
counts = {label: sum(value["correct"] == label for value in verdicts.values())
          for label in ("CORRECT", "PARTLY CORRECT", "WRONG")}
print_table(["Check", "Count out of 10"], [
    ("Correct", str(counts["CORRECT"])), ("Partly correct", str(counts["PARTLY CORRECT"])),
    ("Wrong", str(counts["WRONG"])),
    ("Supported by the chunks", str(sum(value["supported"] == "YES" for value in verdicts.values())))],
    title="D2.1 complete summary")
_ = (OUT_DIR / "answers_and_verdicts.json").write_text(
    json.dumps({"generator": GENERATOR_ID, "judge": JUDGE_MODEL_ID, "queries": QUERIES, "chunks": final_chunks,
                "assignment1_answers": a1_answers, "assignment1_ooc_answer": a1_ooc_answer,
                "answers": answers, "verdicts": verdicts,
                "ooc_query": OOC_QUERY, "ooc_answer": ooc_answer}, indent=2, ensure_ascii=False), encoding="utf-8")

**D2.1 judge results**

| Query | Correct | Supported | Reason |
| --- | --- | --- | --- |
| Q1: What should be considered when planning detection of undiagnosed T2DM and prevention programmes in low- and middle-income countries? | CORRECT | YES | The model answer accurately reflects the expected answer, and all its statements can be found in the provided chunks. |
| Q2: Which foods should people with diabetes and dyslipidaemia restrict because of their saturated and trans fats? | PARTLY CORRECT | YES | The model answer partially states the key fact that foods sources of saturated and trans fats like vanaspati, butter, ghee, margarine, coconut oil, red meats like sausages, ham, bacon, egg yolk, whole milk, and its products should be restricted, but it also includes additional information not present in the expected answer. |
| Q3: How much flax seed can vegetarians take daily to obtain omega-3 fatty acids? | CORRECT | YES | The model answer states the same key fact as the expected answer, and every statement in the model answer can be found in the provided chunks. |
| Q4: Why does WHO conduct a periodic survey of country capacity for noncommunicable diseases? | CORRECT | YES | The model answer states the same key fact as the expected answer, which is that WHO conducts a periodic survey of country capacity for noncommunicable diseases. |
| Q5: How should glycaemic targets be set for people with T2DM? | CORRECT | YES | Every statement in the model answer can be found in the provided chunks. |
| Q6: How many data sources underpinned the IDF diabetes prevalence estimates, and what period did the research cover? | PARTLY CORRECT | YES | The model answer states that the research covered a period that included studies up to 2020, with the exclusion of studies conducted prior to 2005, except in specific circumstances where no other credible in-country data was available, which is partially correct as it does not mention the specific period of 2005-2020. |
| Q7: How should people starting a sulphonylurea be counselled about hypoglycaemia warning signs? | PARTLY CORRECT | NO | The model answer states "including early signs such as tremor, sweating, shaking, and irritability", but the chunks only mention "early signs" without specifying the symptoms, and the chunks mention "irritability" but not in the context of early signs. |
| Q8: How should the diet be changed during fever or other illness in diabetes? | CORRECT | YES | The model answer and expected answer both state that the consistency and texture of the diabetic diet should be modified during fever or other illness to maintain adequate calorie intake. |
| Q9: For how many days is each type of continuous glucose monitoring used when advanced monitoring is needed? | PARTLY CORRECT | NO | The model answer states "For 14 days in the case of Continuous Glucose Monitoring (CGM) [1], and intermittently scanned CGM (iSCGM) for 14 days [1]" but the correct answer is "Continuous Glucose Monitoring (CGM) is applied for 5-7 days, whereas intermittently scanned CGM (iSCGM) is used for a longer period of 14 days". |
| Q10: Which foods and fluids should be encouraged for people with diabetes on sick days? | CORRECT | YES | Every statement in the model answer can be found in the provided chunks. |

D2.1 complete summary
Check                     Count out of 10
-----------------------------------------
Correct                                 6
Partly correct                          4
Wrong                                   0
Supported by the chunks                 8


39526

#### D2.2 Spot-check the judge (1 mark)

The next cell displays three cases with different verdicts where available.

In [42]:
# D2.2: select three queries for the human spot-check.
SPOT_QIDS = []
seen_verdicts = set()
for item in QUERIES:
    value = verdicts[item["qid"]]
    signature = (value["correct"], value["supported"])
    if signature not in seen_verdicts and len(SPOT_QIDS) < 3:
        SPOT_QIDS.append(item["qid"])
        seen_verdicts.add(signature)
for item in QUERIES:
    if len(SPOT_QIDS) == 3:
        break
    if item["qid"] not in SPOT_QIDS:
        SPOT_QIDS.append(item["qid"])
for qid in SPOT_QIDS:
    item = next(value for value in QUERIES if value["qid"] == qid)
    print("=" * 80)
    print(f"{qid}: {item['query']}")
    print(f"Expected: {item['expected_answer']}")
    print(f"Answer: {answers[qid]}")
    for number, chunk in enumerate(final_chunks[qid], 1):
        print(f"\n[{number}] {chunk['chunk_id']}\n{chunk['text']}")
    value = verdicts[qid]
    print(f"\nJudge: {value['correct']} / {value['supported']}\nReason: {value['reason']}")


Q1: What should be considered when planning detection of undiagnosed T2DM and prevention programmes in low- and middle-income countries?
Expected: Planners should expect that capacity to detect undiagnosed T2DM and intermediate hyperglycaemia and to implement diabetes prevention programmes varies considerably between countries, health services and systems, and the variation is especially pronounced in low- and middle-income countries.
Answer: When planning detection of undiagnosed T2DM and prevention programmes in low- and middle-income countries, it is important to consider the varying capacities to detect individuals with undiagnosed T2DM and impaired glucose tolerance (IGT) and implement diabetes prevention programmes [1]. Additionally, there is a need to address the uneven implementation and funding of national policies and guidelines [3], as well as the limited availability of basic technologies for early detection, diagnosis, and monitoring of diabetes in primary care settings [1

**D2.2 Human verdicts.** Enter independent decisions in `SPOT_REVIEW`, keyed by the three displayed query numbers: `correct` (`CORRECT`, `PARTLY CORRECT`, or `WRONG`), `supported` (`YES` or `NO`), and `reason` based on the reference answer and chunks. Quote relevant evidence when disagreeing with the judge. Agreement counts use your entries.

In [43]:
# D2.2: enter the human spot-check judgments.
SPOT_REVIEW = {}

spot_rows, spot_pending, spot_disagreements = [], [], []
spot_agreements = 0
for qid in SPOT_QIDS:
    human_review = SPOT_REVIEW.get(qid, {})
    # Missing reviews stay pending; never copy the judge's labels into them.
    if (
        human_review.get("correct") not in {"CORRECT", "PARTLY CORRECT", "WRONG"}
        or human_review.get("supported") not in {"YES", "NO"}
        or not str(human_review.get("reason", "")).strip()
    ):
        spot_pending.append(qid)
        continue

    judge = verdicts[qid]
    # Agreement requires matching both correctness and evidence support.
    agrees = (human_review["correct"], human_review["supported"]) == (judge["correct"], judge["supported"])
    spot_agreements += agrees
    spot_rows.append((
        qid,
        f"{judge['correct']} / {judge['supported']}",
        f"{human_review['correct']} / {human_review['supported']}",
        str(agrees),
        human_review["reason"],
    ))
    if not agrees:
        spot_disagreements.append(f"{qid}: {human_review['reason']}")

if spot_rows:
    show_md_table(
        ["Query", "Judge", "Human", "Agree on both checks", "Human explanation"], spot_rows,
        title="D2.2 human spot-check",
    )
if spot_pending:
    print(f"Human review pending: {', '.join(spot_pending)}. Fill SPOT_REVIEW and rerun this cell.")
else:
    display(Markdown(
        f"Agreement: **{spot_agreements} of 3** verdicts. "
        + ("Disagreements: " + " ".join(spot_disagreements) if spot_disagreements else
           "No disagreements.")
    ))

Human review pending: Q1, Q2, Q7. Fill SPOT_REVIEW and rerun this cell.


#### D2.3 Find the cause (1 mark)

Inspect one wrong or unsupported answer and its three chunks. Determine whether the needed facts were available, identify the answer error, and attribute it to retrieval, generation, or both. Literal span matching is only a diagnostic; equivalent wording may still supply the facts.

In [44]:
# D2.3: show the first failure case.

FAILED_QID = next((item["qid"] for item in QUERIES if failed(item["qid"])), None)
if FAILED_QID is None:
    print("No wrong or unsupported answer found, so there is no failure case to inspect.")
else:
    item = next(value for value in QUERIES if value["qid"] == FAILED_QID)
    verdict = verdicts[FAILED_QID]
    print(f"{FAILED_QID}: {item['query']}")
    print(f"Expected: {item['expected_answer']}")
    print(f"Answer: {answers[FAILED_QID]}")
    print(f"Judge: {verdict['correct']} / {verdict['supported']}\nReason: {verdict['reason']}")
    for number, chunk in enumerate(final_chunks[FAILED_QID], 1):
        print(f"\n[{number}] {chunk['chunk_id']}\n{chunk['text']}")
    literal_hit = any(is_hit(chunk["text"], item["span"]) for chunk in final_chunks[FAILED_QID])
    print(f"\nAnnotated span: {item['span']}\nLiteral span present: {literal_hit}")


Q7: How should people starting a sulphonylurea be counselled about hypoglycaemia warning signs?
Expected: They should be told that early warning signs include tremor, sweating, shaking and irritability, while later signs involve poor concentration and impaired co-ordination. Recognising both stages lets them act before the episode becomes severe.
Answer: People starting a sulphonylurea should be counselled about hypoglycaemia warning signs, including early signs such as tremor, sweating, shaking, and irritability, and later signs like lack of concentration and co-ordination [2].
Judge: PARTLY CORRECT / NO
Reason: The model answer states "including early signs such as tremor, sweating, shaking, and irritability", but the chunks only mention "early signs" without specifying the symptoms, and the chunks mention "irritability" but not in the context of early signs.

[1] sign154-pharmacological-glycaemic-control#378
, polyuria
and weight loss. They are more effective early in the management

**Human cause analysis.** Complete `FAILURE_REVIEW` for the selected answer: `qid`, `needed_information_present` (a Python Boolean), `cause` (`retrieval`, `generation`, or `both`), `evidence` (a relevant quote or missing fact), and `explanation`. Unsupported extra claims may indicate generation failure even when retrieval also missed facts.


In [45]:
# D2.3: the human cause analysis for the failed answer, and the check that it is complete.
FAILURE_REVIEW = {
    "qid": "",
    "needed_information_present": None,
    "cause": "",
    "evidence": "",
    "explanation": "",
}

cause_complete = (
    FAILED_QID is not None
    and FAILURE_REVIEW["qid"] == FAILED_QID
    and type(FAILURE_REVIEW["needed_information_present"]) is bool
    and FAILURE_REVIEW["cause"] in {"retrieval", "generation", "both"}
    and all(
        isinstance(FAILURE_REVIEW[field], str) and FAILURE_REVIEW[field].strip()
        for field in ("evidence", "explanation")
    )
)
if cause_complete:
    availability = "was present" if FAILURE_REVIEW["needed_information_present"] else "was not fully present"
    display(Markdown(
        f"**D2.3 ({FAILED_QID}):** The needed information {availability} in the retrieved chunks. "
        f"Cause: **{FAILURE_REVIEW['cause']}**. {FAILURE_REVIEW['explanation']} "
        f"Evidence: {FAILURE_REVIEW['evidence']}"
    ))
else:
    print("D2.3 pending: fill FAILURE_REVIEW and rerun this cell.")


D2.3 pending: fill FAILURE_REVIEW and rerun this cell.


#### D2.4 Suggest a fix (1 mark)

Brief: "Suggest a fix: propose one change that would fix it, for example a different chunk size, search method, reranking or prompt wording, and explain why it would help. (1 mark)"

Complete `FIX_REVIEW` with `fix` and `why_it_helps`: one concrete change linked to the observed failure. For example, a boundary-split answer may motivate changing the allowed chunk size. Treat the fix as a proposal unless tested.

In [46]:
# D2.4: the proposed fix for the failed answer, and the check that it is complete.
FIX_REVIEW = {
    "fix": "",
    "why_it_helps": "",
}

fix_complete = FAILED_QID is not None and all(
    isinstance(FIX_REVIEW[field], str) and FIX_REVIEW[field].strip()
    for field in ("fix", "why_it_helps")
)
if fix_complete:
    display(Markdown(
        f"**D2.4 proposed fix:** {FIX_REVIEW['fix']} "
        f"Why it should help: {FIX_REVIEW['why_it_helps']}"
    ))
else:
    print("D2.4 pending: fill FIX_REVIEW and rerun this cell.")


D2.4 pending: fill FIX_REVIEW and rerun this cell.


### Completion checks

Confirms that the reviewed queries, judge verdicts, spot-checks, failure analysis and the out-of-corpus refusal are complete.

In [47]:
# Completion checks for the notebook and the saved human review.
completion_checks = {
    "Ten reviewed evaluation query/span pairs": (
        len(QUERIES) == 10 and all(item.get("human_reviewed") is True for item in QUERIES)
    ),
    "Ten valid judge verdicts": (
        set(verdicts) == {item["qid"] for item in QUERIES}
        and all(valid_verdict(value) for value in verdicts.values())
    ),
    "Three completed human spot-checks": len(SPOT_QIDS) == 3 and not spot_pending,
    "Evidence-based failure analysis and proposed fix": cause_complete and fix_complete,
    "Out-of-corpus refusal demonstrated": is_not_found(ooc_answer),
    "All non-refusal answers have valid citation numbers": all(
        is_not_found(answer) or has_valid_citation(answer) for answer in answers.values()
    ),
}
print_table(
    ["Completion check", "Status"],
    [(label, "PASS" if complete else "PENDING") for label, complete in completion_checks.items()],
    title="Notebook content checks",
)
_ = (OUT_DIR / "human_review.json").write_text(
    json.dumps({
        "spot_checks": SPOT_REVIEW,
        "failure_review": FAILURE_REVIEW,
        "fix_review": FIX_REVIEW,
        "completion_checks": completion_checks,
    }, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

Notebook content checks
Completion check                                       Status
-------------------------------------------------------------
Ten reviewed evaluation query/span pairs                 PASS
Ten valid judge verdicts                                 PASS
Three completed human spot-checks                     PENDING
Evidence-based failure analysis and proposed fix      PENDING
Out-of-corpus refusal demonstrated                       PASS
All non-refusal answers have valid citation numbers      PASS
